# C-test ONE · Annotator

*Ctest-Annotator-v2.28.0*

Certifies the items that are unquestionable under one reference machine: runs
candidate descriptions (the empty description, hints, equivalents and the seed) conditionally on
the prefix, and records which reproduce the continuation, their conditional cost and the tokens
used. From these come $\widehat{K}^{|}$, $\tau$ and $\widehat{Kt}^{|}$.

| reads | writes |
|---|---|
| `ctest_battery.json`, `ctest_programs.json`, `ctest_pricing_<machine>.json` | `ctest_metrics.json` |

**Needs** a GPU, a Gemini API key for the judges and the hint composer, and, for gated models, a
Hugging Face token. The slowest stage: tens of minutes per item at the hard end. The machine is
`MACHINE_ID`.

## Running it

**Locally.** From the repository root, install the requirements and open the notebook in Jupyter:

```
pip install -r requirements.txt
jupyter lab notebooks/
```

API keys are read from environment variables (see `.env.example`). On the first local run the
notebook copies `data/` into a `work/` folder and runs there, so the released data are never
modified; set `CTEST_WORK_DIR` to use another folder.

**Colab.** Upload the data files to `MyDrive/ctest` and mount Drive; keys go in the Secrets panel.

**Kaggle.** Attach a dataset holding the data files and set `KAGGLE_DATASET` to its name; keys go in
Add-ons, Secrets. With Kaggle credentials the notebook can also push its results back as a new
version of the dataset.


In [ ]:
# Local runs work in a copy of the data, so that data/ stays exactly as released.
# On Colab and Kaggle this cell does nothing.
import os, glob, shutil
if not (os.path.exists("/kaggle") or os.path.exists("/content")):
    _root = os.getcwd()
    if os.path.basename(_root) == "notebooks":
        _root = os.path.dirname(_root)
    try:
        from dotenv import load_dotenv          # optional: keys from a .env file
        load_dotenv(os.path.join(_root, ".env"))
    except ImportError:
        pass
    WORK_DIR = os.path.abspath(os.environ.get("CTEST_WORK_DIR", os.path.join(_root, "work")))
    os.makedirs(WORK_DIR, exist_ok=True)
    for _f in glob.glob(os.path.join(_root, "data", "*")):
        _dst = os.path.join(WORK_DIR, os.path.basename(_f))
        if not os.path.exists(_dst):
            shutil.copy(_f, _dst)
    os.chdir(WORK_DIR)
    print(f"local run: working in {WORK_DIR}")


## 1 · Install

Run once.
On Colab a restart is not normally needed.


In [ ]:
%pip install -q -U transformers accelerate bitsandbytes

# Mount Drive here, so the preamble never has to stop for it. This is a no-op if
# it is already mounted.
try:
    import google.colab
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as _e:
    print("not on Colab, or Drive not mounted:", _e)


## 2 · Preamble

Checks the environment before anything expensive happens: interpreter, package
versions, GPU and its memory, free disk, and which input files were found.
It
stops here if the battery is missing or there is no GPU.


In [ ]:
import os, sys, re, json, math, time, glob, shutil, hashlib, unicodedata, platform, datetime
import statistics as st, collections

BUILD = "Ctest-Annotator-v2.28.0"
print("=" * 72)
print(f"{BUILD}   (S2 of the C-test pipeline)")
print("=" * 72)

if os.path.exists("/kaggle"):
    ENV = "kaggle"
elif os.path.isdir("/content"):
    ENV = "colab"
else:
    ENV = "local"
# Two separate places, because they are separate things.
DRIVE_FOLDER   = "ctest"        # Colab: /content/drive/MyDrive/ctest
KAGGLE_DATASET = "ctest-one-data"    # Kaggle: the dataset to read and push
                                # to. A Kaggle dataset is a FOLDER of
                                # files, not a single file.
                                #
                                # Only the DEFAULT. A secret of the same name
                                # wins, resolved below once _secret() exists.
                                # The username already comes from a secret, so
                                # the dataset should too: this notebook is then
                                # byte-identical in every account and there is
                                # nothing to edit when moving between them.

# On Kaggle a dataset is READ-ONLY at /kaggle/input, and the working
# directory is already where files are written, so copying there would
# copy a file onto itself and raise SameFileError. OUTDIR stays "." and
# persistence is the dataset push in the cell below.
DRIVE = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
OUTDIR = DRIVE if (ENV == "colab" and os.path.isdir("/content/drive/MyDrive")) else "."
print(f"environment      {ENV}   python {platform.python_version()}")

def stage(t):
    """A run heading. The persistence cell has always called this and it was
    never defined anywhere in this notebook: the cell died before reaching it,
    first on the Drive guard and then on _secret, so the missing name was only
    ever the third error in the queue."""
    print(f"\n{'='*72}\n== {t}\n{'='*72}", flush=True)

def log(*a): print(*a, flush=True)

def find(f):
    for p in [f, f"/content/{f}", f"{DRIVE}/{f}"] + glob.glob(f"/kaggle/input/**/{f}", recursive=True):
        if os.path.exists(p): return p
    return None

def _secret(n):
    if os.environ.get(n): return os.environ[n]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(n)
    except Exception: pass
    try:
        from google.colab import userdata
        return userdata.get(n)
    except Exception: return None

_key = _secret("GOOGLE_API_KEY")
if _key: os.environ["GOOGLE_API_KEY"] = _key
print(f"GOOGLE_API_KEY   {'found' if _key else 'MISSING, the judge will be off'}")

# The dataset, from a secret when there is one. Two accounts can then work in
# parallel on separate copies without either notebook differing by a character:
# each account's KAGGLE_DATASET secret names its own.
_ds = _secret("KAGGLE_DATASET")
if _ds:
    KAGGLE_DATASET = _ds.strip()
print(f"KAGGLE_DATASET   {KAGGLE_DATASET}"
      + ("   from the secret" if _ds else "   the built-in default"))

# The Hub token has to be put INTO the environment before transformers is asked
# for a model. A Kaggle secret is not an environment variable: it lives behind
# UserSecretsClient and is invisible to huggingface_hub until something fetches
# it, which is why a notebook with HF_TOKEN in its secrets still reports
# "unauthenticated requests to the HF Hub". Unauthenticated still works for a
# public model; it is rate limited and slower, and it fails outright on a gated
# one.
#
# Both names are set. huggingface_hub reads HF_TOKEN, and some versions of
# transformers and datasets still look for HUGGING_FACE_HUB_TOKEN.
_hf = _secret("HF_TOKEN") or _secret("HUGGING_FACE_HUB_TOKEN") \
      or _secret("HUGGINGFACE_TOKEN")
if _hf:
    os.environ["HF_TOKEN"] = _hf
    os.environ["HUGGING_FACE_HUB_TOKEN"] = _hf
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = _hf
    print(f"HF_TOKEN         found, {len(_hf)} chars, set for the Hub")
else:
    print( "HF_TOKEN         not found. Downloads are unauthenticated: rate")
    print( "                 limited, and gated models will not resolve. On")
    print( "                 Kaggle add it under Add-ons -> Secrets with the")
    print( "                 toggle ON for this notebook; on Colab use the key")
    print( "                 panel. The secret must be named HF_TOKEN.")

# shutil.copy writes a FILE when its destination is not an existing directory,
# so without this every save landed in a file named after the folder, and the
# results went with the session. Make the directory, and check it is one.
if OUTDIR != ".":
    if os.path.exists(OUTDIR) and not os.path.isdir(OUTDIR):
        raise SystemExit(f"{OUTDIR} exists but is a file, not a folder. It is "
                         f"probably an earlier run's metrics written there by "
                         f"mistake. Rename or delete it, then run this again.")
    os.makedirs(OUTDIR, exist_ok=True)
elif ENV == "kaggle":
    print("                 (see the persistence cell: /kaggle/working is "
          "wiped between versions unless pushed)")
elif ENV == "colab":
    raise SystemExit("Drive is not mounted, so anything written would be lost "
                     "when the session ends. Run this first, in a cell above:\n"
                     "    from google.colab import drive\n"
                     "    drive.mount('/content/drive')")
print(f"outputs go to    {OUTDIR}")

import torch, transformers
print(f"torch            {torch.__version__}")
print(f"transformers     {transformers.__version__}")
try:
    import bitsandbytes; print(f"bitsandbytes     {bitsandbytes.__version__}")
except Exception as e:
    print(f"bitsandbytes     MISSING ({e}); 4-bit loading will fail")

if not torch.cuda.is_available():
    print("GPU              NONE. Runtime > Change runtime type > GPU")
    raise SystemExit("a GPU is required")
_gp = torch.cuda.get_device_properties(0)
VRAM = _gp.total_memory / 1e9
print(f"GPU              {_gp.name}, {VRAM:.0f} GB"
      + ("   WARNING: an 8B model in 4-bit may not fit" if VRAM < 13 else ""))
_du = shutil.disk_usage(".")
print(f"disk free        {_du.free/1e9:.0f} GB"
      + ("   WARNING: the download needs about 20 GB" if _du.free < 25e9 else ""))


## 3 · Config

`K_MAX` sets how many terms a program must reproduce, and also how long the
listing has to be, so it controls both how strict certification is and how
expensive `p_lam` becomes.
`TOP_N` bounds the certification cost per item, since
programs are tried cheapest first and the search stops at the first that
works.


In [ ]:
# ============================ CONFIG =======================================
# Machines differ in ways that break things quietly: the marker that closes
# their reasoning, whether the chat template takes an argument to switch
# thinking on, and whether they are meant to be decoded greedily. OLMo loops
# forever under greedy decoding and answers cleanly under its own sampling
# settings, so these belong to the machine and not to the run.
MACHINES = {
  "qwen3-8b-4bit-think": dict(
      model="Qwen/Qwen3-8B", quant="4bit-nf4", think_end="</think>",
      template_kw=dict(enable_thinking=True),
      # Qwen3 closes its reasoning, so the answer is what follows the marker.
      # Its working is not its answer, and stopping on terms found there cut
      # generations off mid-thought.
      stop_on_extract=False,
      budgets=[1500, 3000, 6000], hint_budgets=[1500, 3000],
      extend_on_complete=False,
      # OFF for the same reason as the ladder: the fifty-four items already
      # measured were done by trying three candidates and stopping at the
      # first that worked, and turning either on would make a new item
      # incomparable with them.
      prune_above_bar=False, keep_going_after_success=False, top_n=3,
      gen=dict(do_sample=False, temperature=None, top_p=None, top_k=None)),
  "qwen3-14b-4bit-think": dict(
      # The second reference machine. Same family and template as the 8B, so
      # nothing about the prompting changes and any difference in what
      # certifies is the model rather than the harness. About 9 GB in 4-bit,
      # so it still fits a T4, and generation is roughly 1.7 times slower.
      model="Qwen/Qwen3-14B", quant="4bit-nf4", think_end="</think>",
      template_kw=dict(enable_thinking=True),
      stop_on_extract=False,
      budgets=[3000, 6000, 12000], hint_budgets=[3000, 6000],
      extend_on_complete=True,
      prune_above_bar=True, keep_going_after_success=True, top_n=0,
      gen=dict(do_sample=False, temperature=None, top_p=None, top_k=None)),
  "olmo3-7b-4bit-think": dict(
      model="allenai/Olmo-3-7B-Think", quant="4bit-nf4", think_end="</think>",
      template_kw={},                       # its template opens <think> already
      # OLMo never closes its reasoning, so there is nothing after a marker to
      # read and the terms in the working are the only answer there is. Its
      # template ends turns with <|im_end|>, which is not in the tokenizer's
      # special tokens map, so that has to be passed explicitly or the model has
      # no reason to stop at the end of its own turn.
      stop_on_extract=True, extra_eos=["<|im_end|>"],
      budgets=[1500, 3000, 6000], hint_budgets=[1500, 3000],
      extend_on_complete=True,
      gen=dict(do_sample=True, temperature=0.6, top_p=0.95)),
  "phi-4-reasoning": dict(
      # A THIRD REFERENCE MACHINE, from a family unrelated to Qwen, so that a
      # difference between two machines is not automatically a difference of
      # size within one lineage.
      #
      # The tag carries no 4bit-think suffix because the unquestionability
      # pass derives it from the model name and wrote it plain; the Qwen tags
      # carry one only because their file names were set by hand.
      model="microsoft/Phi-4-reasoning", quant="4bit-nf4",
      # Phi-4-reasoning closes its reasoning with </think>, as Qwen3 does. IF
      # EVERY GENERATION SCORES ZERO, check this first: the extract would then
      # be reading the working rather than the answer.
      think_end="</think>",
      # `enable_thinking` is a Qwen-specific template argument and raises on
      # other families, so nothing is passed here.
      template_kw=dict(),
      stop_on_extract=False,
      budgets=[1500, 3000, 6000], hint_budgets=[1500, 3000],
      extend_on_complete=False,
      prune_above_bar=False, keep_going_after_success=False, top_n=3,
      gen=dict(do_sample=False, temperature=None, top_p=None, top_k=None)),
}
MACHINE_ID = "qwen3-14b-4bit-think"
MACHINE      = MACHINES[MACHINE_ID]["model"]
THINK_END    = MACHINES[MACHINE_ID]["think_end"]
GEN_KW       = MACHINES[MACHINE_ID]["gen"]
TEMPLATE_KW  = MACHINES[MACHINE_ID]["template_kw"]
STOCHASTIC   = bool(GEN_KW.get("do_sample"))
STOP_ON_EXTRACT = MACHINES[MACHINE_ID].get("stop_on_extract", False)
EXTRA_EOS    = MACHINES[MACHINE_ID].get("extra_eos", [])
PRICING_FILE = None   # None derives it from MACHINE, as S1b names it
ANNOTATE_PREFIX = "v17"   # batches to draw from, NEWEST FIRST. The
                          # whole of the first is taken before any of the
                          # second: a run is about how the CURRENT generator
                          # behaves, and older batches were made under different
                          # dials and lengths, so they are filler rather than
                          # evidence. A plain string works too, and "" means
                          # all of them.
K_MAX        = 20     # k = min(K_MAX, terms available after the prefix).
                      # S1b leaves at least 12 terms after n_M, so k is never
                      # below 12 and this is only a cap.
TOP_N        = 0
RESCUE       = True   # let the judge count terms when the parser cannot
EXTEND_NEW   = 2500   # granted when the judge says the derivation was working
USE_JUDGE    = True   # use the judges at all
BUDGET_JUDGE  = "gemini-3.5-flash-lite"
# SCORING_JUDGE checks hits found by the substring route, which counted "be",
# "go" and "is" appearing in ordinary prose as terms. It is called rarely and
# it changes n_ok, and therefore h, so it uses a stronger model.
SCORING_JUDGE = "gemini-3.6-flash"
SKIP_DONE_IN = None
SKIP_IDS     = []     # no exclusions: old items are controls, not waste
METRICS      = "ctest_metrics.json"
# written by the composer, S1. PROGRAMS holds equivalents and rivals; only the
# equivalents are read here, since rivals price unquestionability and that is
# the composer's business.
BATTERY_FILE  = "ctest_battery.json"
PROGRAMS_FILE = "ctest_programs.json"
# the old monolithic files, still read if the new ones are absent
DEVICE       = "cuda"
VALID_ONLY = True    # ONLY VALID ITEMS, and leave it that way. An item whose
                     # seed does not reproduce its own sequence is discarded by
                     # every table downstream, so annotating it spends an hour
                     # of the reference machine on a number nothing will read.

ANNOTATE_ONLY = []
N_NEW    = 99
N_OLD    = 0
REDO_OLD = False
KEEP_ORDER = False    # the gate's own order
FRESH_FIRST = True

WALL_CLOCK_MIN = 690    # stop cleanly before a 12-hour session ends; 0 for none

STATE_PUSH_EVERY = 1
BUDGETS_OVERRIDE      = None      # e.g. [3000, 6000, 12000, 24000]
HINT_BUDGETS_OVERRIDE = None

BUDGETS      = BUDGETS_OVERRIDE or MACHINES[MACHINE_ID].get(
                   "budgets", [1500, 3000, 6000])
HINT_BUDGETS = HINT_BUDGETS_OVERRIDE or MACHINES[MACHINE_ID].get(
                   "hint_budgets", [1500, 3000])

EXTEND_ON_COMPLETE = MACHINES[MACHINE_ID].get("extend_on_complete", False)

PRUNE_ABOVE_BAR          = MACHINES[MACHINE_ID].get("prune_above_bar", False)
KEEP_GOING_AFTER_SUCCESS = MACHINES[MACHINE_ID].get("keep_going_after_success",
                                                    False)
TOP_N = MACHINES[MACHINE_ID].get("top_n", TOP_N)

# ===========================================================================
print(f"\nmachine          {MACHINE_ID}")
print(f"                 {MACHINE}, {MACHINES[MACHINE_ID]['quant']}")
print(f"decoding         " + ("sampling, seeded per call" if STOCHASTIC else "greedy")
      + f"   {GEN_KW}")
print(f"k                min({K_MAX}, terms available)")
HINTS_FREE     = True     # hints taken from the battery, no calls
HINTS_ASKED    = True     # hints proposed by the composer
# ---- how many certifications an item may spend ---------------------------
# Sorted by cost, the candidates form a threshold: p_eps at the bottom, p_lam
# at the top, and the cheapest string that certifies somewhere between. The
# search is a binary one over that list, so the probes go where the answer is
# rather than always at the bottom.
HINT_PROBES    = 3        # binary-search probes over the hints. Three brackets

CLOSE_EQ_HINT_GAP = True
GAP_MIN_BITS = 50    # only when the rule costs this much more than the dearest
                     # hint
GAP_ASK_N    = 12    # descriptions asked for. Generous, because the composer
                     # cannot price what it writes: it is asked for the middle
                     # ground and the pricing decides what actually lands
                     # there. A small batch can come back clustered on one
                     # phrasing, or miss the band entirely, and another call
                     # costs a few seconds against the half hour a rung takes.
GAP_TRY_N    = 3     # of those landing in the band, how many to run, taken
                     # SPREAD across it rather than cheapest first: the
                     # cheapest sit next to the hints that already failed
                          # eight hints, and an item has five to ten.
PROBE_BELOW    = 2        # after a winner is found, probe this far under it.
                          # Cost and sufficiency agree only as a tendency: on
                          # one run a rule failed at 100.8 bits and another
                          # certified at 103.0, so the first thing that works
                          # is not always the cheapest that would.
HINT_TOP_N     = 3
HINT_ASK_N     = HINT_TOP_N + 2   # how many the composer is ASKED for, which is
                          # more than are certified, on purpose. Pricing a hint
                          # is one forward pass and certifying one is minutes,
                          # so a wider pool costs almost nothing and gives the
                          # cost ordering something to choose from. Only the
                          # cheapest HINT_TOP_N of everything, free and asked
                          # together, are actually run.
HINT_ASK_N     = HINT_TOP_N + 2   # how many the composer is ASKED for, which is
                          # more than are certified, on purpose. Pricing a hint
                          # is one forward pass and certifying one is minutes,
                          # so a wider pool costs almost nothing and gives the
                          # cost ordering something to choose from. Only the
                          # cheapest HINT_TOP_N of everything, free and asked
                          # together, are actually run.
HINT_ASK_ROUNDS = 2       # the second round is told what failed and why
HINT_MAX_CHARS = 90       # longer than this is a rule, not a hint
TAU_MAX_CHARS  = 60000   # of the raw text, from the END, shown to the
                         # judge that locates where the answer was complete

HINTX_ONLY       = False   # the ordinary annotation, not the hint round
HINTX_ITEMS      = []      # used only when HINTX_ONLY is True
HINTX_ROUNDS     = 2       # a second round is told what M produced and why
HINTX_ASK_N      = 5       # hints asked per round
HINTX_RUN_N      = 3       # how many reach M per item, at most
HINTX_MAX_CHARS  = 220     # ABOVE the 90 a hint is held to when it is a
                            # pointer. An item that fell to the listing may
                            # need the mechanism named, not just the table, and
                            # 90 characters cannot do that. A hint at 220 that
                            # certifies still costs a fraction of a listing.
HINTX_SCREEN     = True    # Gemini runs each hint before M does
HINTX_MIN_GAIN = 0.10
HINTX_SCREEN_TERMS = 5   # how many terms the screen asks for. It is a
                         # soundness check: does the note point at the right
                         # thing? Not a difficulty one, which a model stronger
                         # than the reference machine cannot give.
HINTX_SCREEN_OUT  = 8000 # room to derive them. At 2000 the screen ran out
                         # before finishing and called it a failure.
HINTX_SCREEN_REPS = 3
HINTX_ONLY_MISSING = False   # OFF: re-judge an item even if it already carries
                             # a record, since the record may be from a run
                             # that was stopped      # [V(p)]^{1@3}: any one reproducing passes

print(f"programs         "
      + (f"{TOP_N} cheapest tried" if TOP_N else "every candidate tried")
      + (", stop at the first certified" if not KEEP_GOING_AFTER_SUCCESS
         else ", cheapest kept"))
print(f"hints            " + ("battery" if HINTS_FREE else "off")
      + (" + composer" if HINTS_ASKED else "")
      + (f", {HINT_TOP_N} tried at up to {HINT_BUDGETS[-1]} tok, before the programs"
         if (HINTS_FREE or HINTS_ASKED) else ""))
print(f"extend on done   {'yes' if EXTEND_ON_COMPLETE else 'no'}"
      + ("   a program that derived every term and ran out of room gets "
         "another rung" if EXTEND_ON_COMPLETE else
         "   as the rest of the battery was measured"))
print(f"search           "
      + (f"every candidate at or below K + delta" if PRUNE_ABOVE_BAR
         else f"the {TOP_N} cheapest")
      + (", and it keeps looking for a cheaper one after the first works"
         if KEEP_GOING_AFTER_SUCCESS else ", stopping at the first that works"))
print(f"ladder           {BUDGETS}, hints {HINT_BUDGETS}"
      + ("   from the machine" if not BUDGETS_OVERRIDE else
         "   OVERRIDDEN, so this run is not comparable with the rest"))
print(f"certification    one run at the full k; budgets {BUDGETS}, each rung\n                 granted only if the judge sees more terms than last time")
print(f"budget judge     {BUDGET_JUDGE if USE_JUDGE else 'off'}, up to {EXTEND_NEW} tok")
print(f"scoring judge    {SCORING_JUDGE if USE_JUDGE else 'off'}")
print(f"selection        {N_OLD} already-studied items as controls, "
      f"then {N_NEW} fresh")
print(f"target           {N_NEW} fresh items")



## 1c · Persistent state on Kaggle

Reading needs nothing: the loaders glob `/kaggle/input/**` and find the files in
whatever dataset is attached. **Writing** is the problem.
A dataset cannot be
written to directly, and the working directory is wiped between versions, so a
run's output is lost unless it is pushed back as a new version of the dataset.

On Colab this cell does nothing, because Drive is already persistent.

### Setting it up, once

1.
Add Input, Datasets, add **ctest-one-data**.
A Kaggle dataset is a folder of
   files, and adding it makes them readable under `/kaggle/input/ctest-one-data/`.
2.
Add-ons, Secrets: add `KAGGLE_USERNAME` and `KAGGLE_KEY` from your
   `kaggle.json`, and turn the toggle on for this notebook.

With no credentials the notebook still runs and the results still appear in the
version output.
Only state across versions is lost, and the cell says so rather
than failing.

### Why the push works by pattern

`kaggle datasets version` replaces the **whole** dataset with the contents of
the pushed folder, so a file that is in the dataset but not staged is deleted.

So everything matching `ctest_*.json` and `ctest_*.csv` is staged. The log says which were carried.

### When it pushes

A push uploads the whole state, so it does not happen per item.
It happens at
the end of a run, and every `STATE_PUSH_EVERY` items if that is set above zero,
which is worth doing on a long GPU run that might be cut off.


In [ ]:
import subprocess

STATE_PATTERNS = ("ctest_*.json", "ctest_*.csv")

STATE_DATASET = ""
STATE_PUSH_EVERY = 3

def _kaggle_ready():
    if ENV != "kaggle":
        return False
    missing = []
    for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        v = _secret(k)
        if v:
            os.environ[k] = v
        else:
            missing.append(k)
    if missing:
        log(f"  state persistence OFF, missing secret(s): {', '.join(missing)}")
        log("     Add-ons -> Secrets -> add them from your kaggle.json, and turn")
        log("     the toggle on for this notebook. Results still appear in the")
        log("     version output; only state across versions is lost.")
        return False
    return True

KAGGLE_OK = _kaggle_ready()

def _slug():
    if STATE_DATASET:
        return STATE_DATASET
    return f"{os.environ.get('KAGGLE_USERNAME','unknown')}/{KAGGLE_DATASET.lower()}"

def state_files():
    """Every file the push should carry, newest copy of each name.

    Local first, since this run may have just written it. Then whatever is in
    the attached dataset and not local, so another notebook's output survives a
    push from this one."""
    chosen, source = {}, {}
    for pat in STATE_PATTERNS:
        for f in sorted(glob.glob(pat)):
            chosen[os.path.basename(f)] = f
            source[os.path.basename(f)] = "this run"
    for pat in STATE_PATTERNS:
        for f in sorted(glob.glob(f"/kaggle/input/**/{pat}", recursive=True)):
            b = os.path.basename(f)
            if b not in chosen:
                chosen[b] = f
                source[b] = "carried forward"
    return chosen, source

FRESH_DIR = "/kaggle/working/_fresh"

def _dataset_names():
    """Filenames actually IN the dataset, from the refreshed download.

    NOT from /kaggle/input. The mount is fixed when the session starts, so it
    can be older than the dataset, and comparing the mount against a push
    assembled from the mount compares a thing with itself and can never fire.
    That was the whole failure: the dataset held ctest_metrics.json, the mount
    did not, and a push assembled from the mount deleted it.

    If the refresh did not run or failed, this returns nothing and the guard
    stays quiet, because there is no trustworthy list to compare against.
    """
    out = set()
    if not os.path.isdir(FRESH_DIR):
        return out
    for pat in STATE_PATTERNS:
        for f in glob.glob(os.path.join(FRESH_DIR, pat)):
            out.add(os.path.basename(f))
    return out

STATE_REFRESH = True

def state_refresh():
    """Download the current dataset version into the working directory.

    Cheap, a couple of megabytes, and it removes a whole class of confusion:
    after this the notebook is reading what is IN the dataset rather than what
    happened to be mounted.

    Never raises. A failure leaves /kaggle/input in place, which is the old
    behaviour, and says so.
    """
    if not (STATE_REFRESH and KAGGLE_OK):
        return
    slug = _slug()
    d = FRESH_DIR
    shutil.rmtree(d, ignore_errors=True); os.makedirs(d, exist_ok=True)
    try:
        r = subprocess.run(["kaggle", "datasets", "download", "-d", slug,
                            "-p", d, "--unzip"],
                           capture_output=True, text=True, timeout=300)
        if r.returncode != 0:
            log(f"  could not refresh from {slug}: "
                f"{(r.stderr or r.stdout).strip()[:120]}")
            log(f"  falling back to whatever the session mounted, which may be "
                f"older than the dataset.")
            return
    except Exception as e:
        log(f"  could not refresh from {slug}: {str(e)[:100]}")
        return

    got, stale = [], []
    for pat in STATE_PATTERNS:
        for f in sorted(glob.glob(os.path.join(d, pat))):
            b = os.path.basename(f)
            mounted = None
            for p in glob.glob(f"/kaggle/input/**/{b}", recursive=True):
                mounted = p; break
            if os.path.exists(b):
                continue
            shutil.copy(f, b)
            got.append(b)
            if mounted is None:
                stale.append(f"{b} (NOT mounted at all)")
            elif os.path.getsize(mounted) != os.path.getsize(f):
                stale.append(f"{b} (mounted copy is a different size)")
    log(f"  refreshed {len(got)} file(s) from {slug}: {', '.join(sorted(got))}")
    if stale:
        log(f"  the session's mount was OUT OF DATE for: {', '.join(stale)}")
        log(f"  That is why this matters: without the refresh those would have "
            f"been read stale or not at all.")
def state_save(msg="update"):
    """Push a new version of the state dataset. Never raises."""
    if not KAGGLE_OK:
        return
    chosen, source = state_files()
    if not chosen:
        log("  no state files to push yet"); return
    d = "/kaggle/working/_state"
    shutil.rmtree(d, ignore_errors=True); os.makedirs(d, exist_ok=True)
    for b, f in chosen.items():
        shutil.copy(f, os.path.join(d, b))
    _in_dataset = _dataset_names()
    if not _in_dataset:
        log(f"  note: no refreshed copy of the dataset, so the push cannot be "
            f"checked for deletions. Set STATE_REFRESH = True.")
    _would_lose = _in_dataset - set(chosen)
    if _would_lose:
        log(f"  REFUSING TO PUSH. These are in the dataset and not in this "
            f"push, so the push would DELETE them:")
        for b in sorted(_would_lose):
            log(f"      {b}")
        log(f"  Most likely the session mounted an old version of the dataset. "
            f"STATE_REFRESH pulls the current one at startup; if that failed,")
        log(f"  the log above says why. Nothing has been pushed and nothing is "
            f"lost.")
        return

    carried = [b for b, s in source.items() if s == "carried forward"]
    log(f"  staging {len(chosen)} files"
        + (f", {len(carried)} carried forward from the dataset: "
           f"{', '.join(sorted(carried)[:4])}"
           f"{' ...' if len(carried) > 4 else ''}" if carried else ""))
    slug = _slug()
    json.dump({"title": slug.split("/")[-1], "id": slug,
               "licenses": [{"name": "CC0-1.0"}]},
              open(f"{d}/dataset-metadata.json", "w"))
    try:
        r = subprocess.run(["kaggle", "datasets", "version", "-p", d, "-m", msg,
                            "-r", "zip", "--dir-mode", "zip"],
                           capture_output=True, text=True, timeout=900)
        out = (r.stdout + r.stderr).strip()
        last = out.split("\n")[-1][:160] if out else ""
        if r.returncode == 0 and "error" not in out.lower():
            log(f"  state push: {last}")
        else:
            log(f"  the version push did not go through. Its output was:")
            for _l in out.split("\n")[-6:]:
                if _l.strip(): log(f"    {_l.strip()[:150]}")
            # Only create when the dataset really is absent. "Already in use"
            # means it exists and the version push failed for some other
            # reason, so creating it cannot help.
            missing = ("404" in out or "not found" in out.lower()) and \
                      "already in use" not in out.lower()
            if missing:
                r2 = subprocess.run(["kaggle", "datasets", "create", "-p", d,
                                     "-r", "zip", "--dir-mode", "zip"],
                                    capture_output=True, text=True, timeout=900)
                o2 = (r2.stdout + r2.stderr).strip()
                if "already in use" in o2.lower():
                    log(f"  the dataset exists after all, so the version push "
                        f"failed for another reason. NOTHING WAS PUSHED: "
                        f"download the files from this version's output.")
                else:
                    log(f"  created dataset '{slug}'. Attach it as an input so "
                        f"the next version can read it back.")
                    log(f"  state push: {o2.split(chr(10))[-1][:160]}")
            else:
                log(f"  NOTHING WAS PUSHED. The files are in this version's "
                    f"output and can be downloaded from there.")
    except Exception as e:
        log(f"  state push failed ({str(e)[:80]}). The results are still in this"
            f" version's output.")

stage("PERSISTENT STATE")
if ENV != "kaggle":
    log(f"  not on Kaggle, so nothing to do. Outputs go to {OUTDIR}.")
elif KAGGLE_OK:
    subprocess.run(["pip", "install", "-q", "kaggle"], capture_output=True)
    log(f"  credentials found for '{os.environ['KAGGLE_USERNAME']}'")
    log(f"  state dataset: {_slug()}"
        + ("   <- NOT the default; this account has its own copy"
           if STATE_DATASET else ""))
    log(f"  pushes" + (f" every {STATE_PUSH_EVERY} items and" if STATE_PUSH_EVERY
        else "") + " at the end of the run")
_attached = sorted({os.path.basename(os.path.dirname(f))
                    for f in glob.glob("/kaggle/input/*/*")})
if ENV == "kaggle":
    if _attached:
        log(f"  datasets attached: {', '.join(_attached)}")
    else:
        log(f"  NO dataset attached. Add Input -> Datasets -> {KAGGLE_DATASET},")
        log(f"  or the loaders will not find the battery.")
_ch, _src = state_files()
if _ch:
    log(f"  state would carry {len(_ch)} files: {', '.join(sorted(_ch)[:6])}"
        + (" ..." if len(_ch) > 6 else ""))

state_refresh()


## 1d · What the run cost

One row per run, appended to `ctest_runs.csv`.
Wall clock, CPU seconds, GPU
seconds and device, and tokens by model split into prompt, answer and thinking.

Nothing here is a measurement of energy.
It is the record from which energy and
CO2 can be worked out later, from published per-token figures and device power,
so those numbers can be revised without re-running anything.
Keeping the inputs
and deriving the rest is the only way that stays honest.

The numbers are approximate on purpose.
CPU time is process time, so it counts
this notebook and not the API's work.
GPU seconds are wall-clock while a model
is loaded, not utilisation.
Thinking tokens are separated because they are most
of the output on a reasoning model and are charged the same, and they are why a
call truncates with nothing usable returned.


In [ ]:

def seed_of(b):
    return b.get("seed") or b.get("rule") or ""

import csv, resource

RUNS_CSV = "ctest_runs.csv"

# Seed the local copy from wherever the file already lives, before anything is
# appended. Every other file is read back through find(), which globs
# /kaggle/input; this one was only ever appended, so on Kaggle it started empty
# each session and each run wrote a fresh one-row file. Worse, the dataset push
# prefers the local copy, so that one row REPLACED the accumulated history on
# every push. Rows were being lost, not merely not added.
_prev = find(RUNS_CSV)
if _prev and os.path.abspath(_prev) != os.path.abspath(RUNS_CSV):
    shutil.copy(_prev, RUNS_CSV)
    with open(RUNS_CSV) as _f:
        _n = max(0, sum(1 for _ in _f) - 1)
    print(f"run history       {_n} earlier run(s) carried in from {_prev}")
elif _prev:
    with open(RUNS_CSV) as _f:
        _n = max(0, sum(1 for _ in _f) - 1)
    print(f"run history       {_n} earlier run(s) already here")
else:
    print(f"run history       none yet, {RUNS_CSV} will be created")

_T_START = time.time()
_RUN_KEY = datetime.datetime.now().isoformat(timespec="seconds")
_UNITS_SO_FAR = [0]        # items finished, updated as the run goes
_RUN_NOTE = [""]           # what the row should say
_CPU_START = resource.getrusage(resource.RUSAGE_SELF).ru_utime + \
             resource.getrusage(resource.RUSAGE_SELF).ru_stime
_GPU_SECONDS = [0.0]      # wall clock with a model loaded; add to it when one is

def _gpu_name():
    try:
        import torch
        if torch.cuda.is_available():
            return torch.cuda.get_device_name(0)
    except Exception:
        pass
    return ""

def account(units=0, note=""):
    """Record this run's cost. Never raises: accounting must not stop work.

    UPSERT, not append. The row is keyed on when the run STARTED, so calling
    this repeatedly updates one row instead of adding several, and the numbers
    only grow. It used to be called once at the end, so a run stopped by hand
    or by the session ending wrote nothing at all and its GPU hours vanished
    from the record. The annotation runs are the long ones and were the ones
    being lost.
    """
    try:
        cpu = (resource.getrusage(resource.RUSAGE_SELF).ru_utime +
               resource.getrusage(resource.RUSAGE_SELF).ru_stime) - _CPU_START
        tk = dict(TOKENS) if "TOKENS" in globals() else {}
        row = dict(
            run_started=_RUN_KEY,
            run_ended=datetime.datetime.now().isoformat(timespec="seconds"),
            build=BUILD, env=ENV, note=note,
            wall_s=round(time.time() - _T_START, 1),
            cpu_s=round(cpu, 1),
            gpu_s=round(_GPU_SECONDS[0], 1),
            gpu=_gpu_name(),
            units=units,
            model=(COMPOSER if "COMPOSER" in globals()
                   else REF_MACHINE if "REF_MACHINE" in globals() else ""),
            api_calls=tk.get("calls", 0),
            prompt_tokens=tk.get("in", 0),
            answer_tokens=tk.get("answer", 0),
            thinking_tokens=tk.get("think", 0),
            output_tokens=tk.get("out", 0),
            forward_passes=(REF.calls if "REF" in globals()
                            and hasattr(REF, "calls") else 0),
        )
        old_rows, old_cols = [], []
        if os.path.exists(RUNS_CSV):
            with open(RUNS_CSV, newline="") as f:
                rd = csv.DictReader(f)
                old_cols = list(rd.fieldnames or [])
                old_rows = list(rd)
        old_rows = [r for r in old_rows
                    if r.get("run_started") != _RUN_KEY]
        cols = list(dict.fromkeys(list(row) + old_cols))
        with open(RUNS_CSV, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=cols)
            w.writeheader()
            for r in old_rows + [row]:
                w.writerow({c: r.get(c, "") for c in cols})
        if OUTDIR != ".":
            shutil.copy(RUNS_CSV, OUTDIR)
    except Exception as e:
        log(f"  accounting failed ({str(e)[:70]}), which changes nothing else")


## 4 · Files and the battery

Items are deduplicated by (`x_pref`, `x_post`): two battery entries that pose
the same test are one item.
Items already studied elsewhere are skipped so this
run annotates fresh ones.


In [ ]:

# the judge runs over the API, so it needs a key. Without one the run still
# works, it just never extends a budget.

print("\nfiles")
# S1b names its pricing file after the machine, so derive it the same way and
# the two notebooks agree without either being told.
if not PRICING_FILE:
    PRICING_FILE = ("ctest_pricing_"
                    + re.sub(r"[^a-z0-9.-]+", "-", MACHINE.split("/")[-1].lower()).strip("-")
                    + ".json")
_need = [(BATTERY_FILE, False), (PROGRAMS_FILE, False),
         (PRICING_FILE, False),
         ]
if SKIP_DONE_IN: _need.append((SKIP_DONE_IN, False))
_paths = {}
for f, required in _need:
    p = find(f); _paths[f] = p
    print(f"  {'found  ' if p else ('MISSING' if required else 'absent ')}  {f}"
          + (f"   {p}" if p else ""))
    if required and not p:
        raise SystemExit(f"{f} is required. Upload it, or put it in {DRIVE}")

# --------------------------------------------------------------- BATTERY ---
# prefer the composer's files, fall back to the old monolithic ones
_bp = _paths.get(BATTERY_FILE)
if not _bp:
    raise SystemExit(f"no battery found. Expected {BATTERY_FILE} from the composer, "
                     f".")
raw = json.load(open(_bp))
BAT = raw["items"] if isinstance(raw, dict) else raw
log(f"battery from {_bp}")

# The program pool. From S1 this is {item: {equivalents, rivals}} and only the
# equivalents are used; the old file was a flat list of priced candidates.
V8 = {}
if _paths.get(PROGRAMS_FILE):
    _pp = json.load(open(_paths[PROGRAMS_FILE])).get("items", {})
    V8 = {i: {"cands": [dict(tag="equivalent", p=e["p"]) for e in v.get("equivalents", [])]}
          for i, v in _pp.items()}
    log(f"equivalents from {_paths[PROGRAMS_FILE]}: {len(V8)} items")
ALREADY = set(SKIP_IDS)
if SKIP_DONE_IN and _paths.get(SKIP_DONE_IN):
    ALREADY |= set(json.load(open(_paths[SKIP_DONE_IN]))["items"])
log(f"\nbattery {len(BAT)} items | program pool for {len(V8)} items | "
    f"{len(ALREADY)} on the skip list")
if not V8:
    log("   WARNING: no program pool. Only the authored rule and the stored\n"
        "   description will be tried, so the rule band will be nearly empty.\n"
        f"   Provide {PROGRAMS_FILE} before relying on these results.")

PRICED = {}
# delta comes with the pricing: it is measured against the machine that priced
# the battery, so it belongs to that file and not to this notebook. Pruning
# needs it, since a candidate dearer than K + delta cannot be chosen.
DELTA = 0.0
_pp = _paths.get(PRICING_FILE)
if _pp:
    try:
        _raw = json.load(open(_pp))
        # The records live under "items". The top level also carries `machine`,
        # `machine_tag`, `delta` and `priced_by`, so reading it directly gave
        # exactly five "records" and every item looked unpriced.
        PRICED = _raw.get("items", _raw) if isinstance(_raw, dict) else {}
        PRICED = {k: v for k, v in PRICED.items()
                  if not k.startswith("_") and isinstance(v, dict)}
        DELTA = float(_raw.get("delta") or 0.0) if isinstance(_raw, dict) else 0.0
        log(f"pricing from {_pp}: {len(PRICED)} records"
            + (f", {sum(1 for v in PRICED.values() if v.get('certified'))} certified"
               if PRICED else ""))
        # A record has an n_M or says why it was skipped. If none of them do,
        # the wrong level of the file is being read and every item will look
        # unpriced, which is a silent empty run rather than an error.
        if PRICED and not any(("n_M" in v) or v.get("skipped")
                              for v in PRICED.values()):
            log(f"   WARNING: none of these look like pricing records. Keys on "
                f"the first: {sorted(list(PRICED.values())[0])[:8]}")
            log(f"   The records should carry n_M. Check the shape of {_pp}.")
    except Exception as e:
        log(f"could not read {PRICING_FILE}: {str(e)[:70]}")
if not PRICED:
    log(f"   WARNING: no pricing file for {MACHINE_ID}. Nothing can be "
        f"annotated: the prefix to show comes from n_M and there is none.")

def is_valid(b):
    """The seed, run cold a second time, reproduced what the first run made.

    RECORDED TWO WAYS and both mean this. Generation writes `valid` as it
    admits an item; the standalone extension pass writes
    `repro.seed_cold.verdict`. Reading only one of them drops half the battery.
    """
    return (b.get("valid") is True
            or (b.get("repro", {}).get("seed_cold") or {}).get("verdict")
            == "reproduced")

seen, ITEMS, _no_price, _not_cert, _invalid = set(), {}, 0, 0, 0
for b in BAT:
    if b.get("excluded") or not b.get("terms"): continue
    if VALID_ONLY and not is_valid(b):
        _invalid += 1; continue
    rec = PRICED.get(b["id"])
    if not rec or rec.get("skipped"):
        _no_price += 1; continue
    n = rec.get("n_M")
    if not n or not rec.get("certified"):
        _not_cert += 1; continue
    avail = len(b["terms"]) - n
    if avail < 8: continue
    k = min(K_MAX, avail)
    x_pref = [str(t) for t in b["terms"][:n]]
    x_post = [str(t) for t in b["terms"][n:n+k]]
    key = (tuple(x_pref), tuple(x_post))
    if key in seen: continue                  # duplicate item, same test
    seen.add(key)
    ITEMS[b["id"]] = dict(id=b["id"], n=n, k=k, x_pref=x_pref, x_post=x_post,
                          rule=b.get("rule_clean") or seed_of(b) or "",
                          desc=b.get("desc") or "",
                          # carried through so the summary can say how solid the
                          # prefix is. An item certified with no composer rival
                          # at that prefix was certified by silence.
                          K=rec.get("K"), slack=rec.get("slack"),
                          # the measured dials, for the selection order. Not
                          # used in any measurement here; they only decide
                          # which items are looked at first, so that a run cut
                          # short still covers the grid and the cost range.
                          r_hat=b.get("r_hat"), t_hat=b.get("t_hat"),
                          p_star_uncond=rec.get("p_star"),
                          tested=rec.get("tested_by_composer"),
                          # material for the hints. h minimises c(p|x_pref)
                          # over every string that certifies, and a hint is
                          # such a string: given the prefix, naming the table
                          # can be enough where the full rule was needed
                          # without it. These cost nothing, they are already
                          # in the battery.
                          theme=b.get("theme"), domain=b.get("domain_real"),
                          mechanism=b.get("mechanism"), gloss=b.get("gloss"),
                          components=b.get("components") or [],
                          cell=(b.get("want_retrieval"), b.get("want_transformation")))
log(f"{len(ITEMS)} items certified and eligible"
    + (f"; {_invalid} not valid" if _invalid else "")
    + (f"; {_no_price} not priced" if _no_price else "")
    + (f"; {_not_cert} priced but not certified" if _not_cert else ""))
_untested = sum(1 for v in ITEMS.values() if v.get("tested") is False)
if _untested:
    log(f"   {_untested} of them certified at a prefix no composer rival was "
        f"proposed against, so that prefix rests on the structural rival alone")


### 4b · Item selection

An explicit list rather than a computed one, so it survives version changes and
can be edited by hand.
The list holds the items to run and, beside each, the
band another machine put it in.
Running a second machine over the same items is
what makes their $h$ values comparable, and the summary reports where the two
disagree.

Edit `ITEM_BANDS` to change the set.
Set a value to `None` for an item with
nothing to compare against.


In [ ]:
# ===========================================================================
#  ITEM SELECTION
#
#  Computed from the pricing file rather than listed by hand. The old list named
#  21 v6 and v8 items from a battery three regimes ago, and hand-listing was
#  right when the pipeline could not say which items were ready. It can now: an
#  item is eligible when it has been priced under this machine and certified,
#  because that is what gives it a defensible prefix to show.
#
#  ANNOTATE_PREFIX picks a batch. The battery holds several regimes at once and
#  their dials and lengths are not comparable, so a run should take one.
#
#  ITEM_BANDS still works and still wins when it is not empty, for re-running a
#  named set under a second machine, which is what makes h_M1 and h_M2
#  comparable.
# ===========================================================================
COMPARE_WITH = MACHINE_ID          # whose earlier bands to compare against
ITEM_BANDS   = {}                  # a named set; empty means compute one

if ITEM_BANDS:
    EXPECTED = dict(ITEM_BANDS)
    SELECTED = [i for i in ITEM_BANDS if i in ITEMS]
    missing  = [i for i in ITEM_BANDS if i not in ITEMS]
    # Later cells read both, so they are defined on this branch too. Leaving
    # them out here meant the run cell failed with a NameError only when a
    # named list was used, which is the path taken least often and so the one
    # least likely to be noticed.
    OLD   = [i for i in SELECTED if EXPECTED.get(i)]
    FRESH = [i for i in SELECTED if not EXPECTED.get(i)]
    log(f"\n{len(SELECTED)} items from an explicit list")
    if missing:
        log(f"   {len(missing)} listed items are not eligible: "
            + ", ".join(missing[:8]))
else:
    # ANNOTATE_PREFIX may be one prefix or a list, NEWEST FIRST. The whole of
    # the first batch is taken before any of the second, because the point of a
    # run is to see how the CURRENT generator behaves; older batches were made
    # under different dials and lengths and are filler, not evidence. They are
    # only reached when the newest batch cannot fill N_NEW.
    _prefixes = ([ANNOTATE_PREFIX] if isinstance(ANNOTATE_PREFIX, str)
                 else list(ANNOTATE_PREFIX or []))
    if not _prefixes:
        _prefixes = [""]
    _by_prefix = []
    for p in _prefixes:
        _by_prefix.append([i for i in ITEMS if not p or i.startswith(p)])
    pool = [i for grp in _by_prefix for i in grp]
    if ANNOTATE_ONLY:
        keep = set(ANNOTATE_ONLY)
        pool = [i for i in pool if i in keep]
        _by_prefix = [[i for i in g if i in keep] for g in _by_prefix]
    _prev = set()
    if os.path.exists(METRICS):
        try:
            _prev = {i for i, v in json.load(open(METRICS)).get("items", {}).items()
                     if v.get(MACHINE_ID, {}).get("complete")}
        except Exception:
            pass
    old = [i for i in pool if i in _prev]

    def _spread(ids, chosen=()):
        """Order so that ANY PREFIX of the result is a good sample.

        Round-robin over cells gives a spread only if a run finishes. Stopped
        early it can still leave a lopsided set, because the cells are visited
        in a fixed order and K is not considered at all.

        This is farthest-point ordering instead. Each item sits at
        (r_hat, t_hat, log K), each axis scaled to its own range so none
        dominates, and the next item taken is the one furthest from everything
        already chosen. Items annotated in an earlier run count as chosen, so
        the run continues the spread rather than starting it again.

        Those three axes because r_hat and t_hat are the dials under test, and
        K is the best proxy for h before h is known, h being K_cond plus the
        cost of the prefix and K_cond being at most K. log K rather than K
        because it runs from 17 to over 300 bits and the top would otherwise
        decide everything.

        The point of it: stop whenever, and the items measured so far still
        cover the grid and the cost range, so a correlation computed on them
        is not the correlation of one corner.
        """
        pts = {}
        for i in ids:
            it = ITEMS[i]
            k = it.get("K") or it.get("K_uncond") or 0
            pts[i] = (it.get("r_hat"), it.get("t_hat"),
                      math.log(k) if k > 0 else None)
        good = [i for i in ids if None not in pts[i]]
        rest = [i for i in ids if i not in good]
        if len(good) < 2:
            return list(ids)

        lo = [min(pts[i][d] for i in good) for d in range(3)]
        hi = [max(pts[i][d] for i in good) for d in range(3)]
        def at(i):
            return tuple((pts[i][d] - lo[d]) / (hi[d] - lo[d])
                         if hi[d] > lo[d] else 0.0 for d in range(3))
        def far(i, seeds):
            return min(sum((a - b) ** 2 for a, b in zip(at(i), at(j)))
                       for j in seeds) if seeds else 0.0

        seeds = [i for i in chosen if i in pts and None not in pts[i]]
        out = []
        pool = list(good)
        if not seeds:
            # start at the extreme, so the first item is not an arbitrary one
            start = max(pool, key=lambda i: sum(at(i)))
            out.append(start); pool.remove(start); seeds = [start]
        while pool:
            nxt = max(pool, key=lambda i: far(i, seeds))
            out.append(nxt); pool.remove(nxt); seeds = seeds + [nxt]
        if rest:
            log(f"   {len(rest)} item(s) have no r_hat, t_hat or K and are "
                f"ordered last: {', '.join(rest[:6])}")
        return out + rest

    FRESH, _taken = [], []
    for p, grp in zip(_prefixes, _by_prefix):
        _pool = [i for i in grp
                 if i not in _prev and i not in ALREADY
                 and i not in set(FRESH)]
        # KEEP_ORDER hands the run the list as written. The spread below is
        # right when the selection is "some items from the battery", since any
        # prefix of it then covers the range. It is wrong when the list IS the
        # instruction: an explicit ANNOTATE_ONLY, ordered so the useful half
        # comes first, was being reshuffled and the run started in the middle
        # of it.
        if KEEP_ORDER and ANNOTATE_ONLY:
            _rank = {x: n for n, x in enumerate(ANNOTATE_ONLY)}
            avail = sorted(_pool, key=lambda i: _rank.get(i, 10**6))
        else:
            avail = _spread(_pool, chosen=list(ALREADY) + list(FRESH))
        room = N_NEW - len(FRESH)
        if room <= 0: break
        FRESH += avail[:room]
        _taken.append((p or "all", min(room, len(avail)), len(avail)))
    OLD   = old[:N_OLD]
    # Which half runs first. Controls first is the default, because when they
    # are a handful of quick re-runs it is worth confirming the settings before
    # spending the night on new items.
    #
    # FRESH_FIRST reverses it, and is right when the re-runs are the
    # EXPERIMENT rather than the check: an unattended run should do the
    # predictable work first, so that a re-run which sits at a raised token cap
    # for half an hour an item cannot eat the whole night before the new items
    # are reached.
    SELECTED = (FRESH + OLD) if FRESH_FIRST else (OLD + FRESH)
    EXPECTED = {i: None for i in SELECTED}
    _sel = [i for i in SELECTED if ITEMS[i].get("r_hat") is not None
            and ITEMS[i].get("K")]
    if _sel:
        _r = [ITEMS[i]["r_hat"] for i in _sel]
        _t = [ITEMS[i]["t_hat"] for i in _sel]
        _k = [ITEMS[i]["K"] for i in _sel]
        log(f"\nordered so that ANY PREFIX is a spread: stop whenever and what "
            f"is measured still covers")
        log(f"  r {min(_r):.1f} to {max(_r):.1f}, t {min(_t):.1f} to "
            f"{max(_t):.1f}, K {min(_k):.0f} to {max(_k):.0f} bits")
        log(f"  On the current battery the first FIVE items already span the "
            f"whole of all three.")

    log(f"\n  order: "
        + ("as written in ANNOTATE_ONLY" if (KEEP_ORDER and ANNOTATE_ONLY)
           else "spread, so any prefix of the run covers the range")
        + ("; the fresh items first" if FRESH_FIRST else
           "; the re-runs first"))
    log(f"\n{len(SELECTED)} items selected: {len(FRESH)} not yet annotated "
        f"under {MACHINE_ID}" + (f", {len(OLD)} re-run" if OLD else ""))
    # THE IDS, IN THE ORDER THEY WILL BE DONE. The count, the cells and the
    # ranges were all printed and none of them says WHICH, so a run that had
    # picked up items nobody wanted looked exactly like one that had not until
    # the first of them started.
    for _n, _i in enumerate(SELECTED, 1):
        _b = ITEMS[_i]
        log(f"   {_n:2d}. {_i}  r{_b.get('r_hat')}t{_b.get('t_hat')}  "
            f"n {_b['n']}  k {_b['k']}  K {_b.get('K') or 0:.0f} bits"
            + ("   RE-RUN" if _i in OLD else ""))
    for p, took, had in _taken:
        log(f"   {p+'*':10s} took {took} of {had} eligible"
            + ("   (none eligible)" if had == 0 else
               "" if took == had else "   (N_NEW reached)"))
    if _taken and _taken[0][2] == 0:
        log(f"   the newest batch has nothing eligible. Either it has not been "
            f"priced yet, or nothing in it certified.")
    if len(_taken) > 1 and _taken[0][2] > 0 and _taken[0][1] == _taken[0][2]:
        log(f"   the newest batch was exhausted, so older ones were used to "
            f"fill. Those were made under different dials and lengths, so read "
            f"them as filler and not as evidence about the current generator.")
    if SELECTED:
        import collections as _c
        log(f"   cells covered: "
            f"{dict(sorted(_c.Counter(ITEMS[i]['cell'] for i in SELECTED).items(), key=lambda x: (x[0] is None, x[0])))}")
        _K = [ITEMS[i]["K"] for i in SELECTED if ITEMS[i].get("K")]
        _n = [ITEMS[i]["n"] for i in SELECTED]
        if _K:
            log(f"   K {min(_K):.0f} to {max(_K):.0f} bits, "
                f"n_M {min(_n)} to {max(_n)}, k {min(ITEMS[i]['k'] for i in SELECTED)} "
                f"to {max(ITEMS[i]['k'] for i in SELECTED)}")
    else:
        log(f"   nothing to do. Either the pricing file has no certified items "
            f"for {ANNOTATE_PREFIX or 'any prefix'}, or they are all done.")


## 5 · The programs

Each item gets `p_eps`, `p_lam`, the authored rule, the stored description, and
whatever proposed rules exist in the pool.
Parse debris and anything too short
to be a program is dropped.


In [ ]:
# ------------------------------------------------------------- PROGRAMS ----
def plausible(p):
    """reject parse debris and anything too short to be a program"""
    if not p or len(p.strip()) < 3: return False
    if not re.search(r"[A-Za-z0-9]", p): return False
    if re.fullmatch(r"[\[\]{}(),.;:\"'\s\\/]+", p.strip()): return False
    return True
def listing_of(x_post): return "then " + ", ".join(x_post)
def pid(p): return hashlib.md5(p.encode()).hexdigest()[:10]

def programs_for(i):
    it = ITEMS[i]
    out = [dict(tag="p_eps", p=""), dict(tag="p_lam", p=listing_of(it["x_post"]))]
    have = {c["p"] for c in out}
    for tag, p in (("seed", it["rule"]), ("stored", it["desc"])):
        if plausible(p) and p not in have: out.append(dict(tag=tag, p=p)); have.add(p)
    for c in V8.get(i, {}).get("cands", []):
        p = c.get("p") or ""
        if c["tag"] in ("empty", "listing") or not plausible(p) or p in have: continue
        out.append(dict(tag=c["tag"], p=p)); have.add(p)
    if HINTS_FREE:
        for c in free_hints(it):
            if c["p"] not in have:
                out.append(c); have.add(c["p"])
    return out


## 6 · The machine

`bits` is one teacher-forced forward pass.
It avoids materialising a full
log-softmax, which would allocate `length × vocab` floats and exhaust a 16 GB
card on a 150k-token vocabulary.

The frames put a program in the same slot for pricing and for execution, so a
program is priced where it will be used.
`c(x_pref | p)` needs the opposite
order, rule before sequence, which is a different frame and a source of the
Bayes residual.


In [ ]:
# --------------------------------------------------------------- MACHINE ---
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# check the type, not just that the name exists: a stray assignment left
# `model` bound to a string once, the guard accepted it, and the loop failed
# deep inside generate()
_have = (hasattr(globals().get("model"), "generate")
         and globals().get("_loaded_machine") == MACHINE)
if _have:
    log(f"\n{MACHINE} is already in memory, not reloading")
else:
  log(f"\nloading {MACHINE} ...")
  _t = time.time()
  tok = AutoTokenizer.from_pretrained(MACHINE)
  if tok.pad_token is None: tok.pad_token = tok.eos_token
  model = AutoModelForCausalLM.from_pretrained(MACHINE, device_map=DEVICE,
      quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
          bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True))
  model.eval()
  _loaded_machine = MACHINE
    # some templates end a turn with a token the tokenizer does not call eos
  EOS_IDS = [i for i in ([tok.eos_token_id] +
                         [tok.convert_tokens_to_ids(t) for t in EXTRA_EOS])
             if isinstance(i, int) and i >= 0]
  log(f"  ending a turn on {EOS_IDS} = {[tok.decode([i]) for i in EOS_IDS]}")
  log(f"loaded in {time.time()-_t:.0f}s, VRAM in use "
      f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

@torch.no_grad()
def bits(text, condition=""):
    """-log2 P(text | condition). Avoids a full log_softmax copy, which would
       allocate batch x length x vocab floats and exhaust a 16 GB card."""
    n_c = tok(condition, return_tensors="pt").input_ids.shape[1] if condition else 1
    ids = tok(condition + text, return_tensors="pt").input_ids.to(DEVICE)
    lg = model(ids).logits[:, :-1]
    tgt = ids[:, 1:]
    tl = (lg.gather(-1, tgt.unsqueeze(-1)).squeeze(-1).float() - lg.logsumexp(-1).float())[0]
    del lg
    return -tl[max(n_c-1, 0):].sum().item() / math.log(2)

# frames: a program is priced in the slot where it will be executed
def F_rule_slot(x_pref): return "Sequence: " + ", ".join(x_pref) + "\nRule:"
def F_seq_alone():       return "Sequence:"
def F_rule_first(p):     return "Rule: " + p + "\nSequence:"
def F_next(x_pref, p):
    return ("Sequence: " + ", ".join(x_pref) + "\nRule:" + ((" " + p) if p else "")
            + "\nNext terms:")
def as_pieces(terms):    return [(" " if j == 0 else ", ") + t for j, t in enumerate(terms)]

CERT = ("Sequence: {seq}\n{rule}Give the next {k} terms.\n"
        "Final line must be exactly: ANSWER: t1, t2, ... (all {k} terms, nothing else)\n")

from transformers import StoppingCriteria, StoppingCriteriaList

# ---- text helpers, needed by the stopping criterion below ----------------
# These were defined after the criterion that calls them, which worked only
# because nothing calls it until generation time. Moved up so the cells can
# be run in any order.
def _fold(w):
    w = unicodedata.normalize("NFKD", str(w))
    return "".join(c for c in w if not unicodedata.combining(c)).strip().strip("\"'.,;:()[]").lower()
def _norm(t): return re.sub(r"\s+", " ", re.sub(r"^(the|a|an)\s+", "", _fold(t))).strip()
def leading_correct(got, x_post):
    n = 0
    for g, t in zip(got, x_post):
        if _norm(g) == _norm(t): n += 1
        else: break
    return n
def after_think(text):
    """Everything the machine wrote after it finished reasoning. The answer has
    to be found here: inside the reasoning it restates the instruction, and
    `ANSWER: t1, t2, ... (all 5 terms, nothing else)` splits on commas into
    enough pieces to look like a finished answer."""
    if not THINK_END: return text
    j = text.rfind(THINK_END)
    return text[j + len(THINK_END):] if j >= 0 else ""

def extract(text, x_post):
    """the answer line, then any working line, then the terms in order in the text"""
    m = re.split(r"ANSWER\s*:", after_think(text), flags=re.I)
    if len(m) > 1:
        got = [x.strip() for x in re.split(r"[,;]",
               re.sub(r"[*`_#]", "", m[-1].strip().split("\n")[0])) if x.strip()]
        n = leading_correct(got, x_post)
        if n: return n, "answer-line"
    best, where = 0, "none"
    for line in text.split("\n"):
        got = [x.strip() for x in re.split(r"[,;]", re.sub(r"[*`_#]", "", line)) if x.strip()]
        for s0 in range(len(got)):
            n = leading_correct(got[s0:], x_post)
            if n > best: best, where = n, "working-line"
            if n: break
    # Last resort: the terms appearing in order anywhere in the text, which
    # catches numbered working. Two guards, because a plain substring search
    # found f, g, h, i inside "figure out the pattern" on a single-letter item,
    # and be, go, is inside ordinary English on another.
    pos, cur, low, at = 0, 0, text.lower(), []
    for t in x_post:
        nt = _norm(t)
        m2 = re.search(r"(?<![a-z0-9])" + re.escape(nt) + r"(?![a-z0-9])", low[pos:])
        if not m2: break
        at.append(pos + m2.start()); pos += m2.end(); cur += 1
    # a real answer has its terms close together, not scattered through prose
    # a derivation works terms out across sentences, so allow room. This only
    # has to reject terms scattered over the whole of a long derivation.
    if cur > 1 and (at[-1] - at[0]) > 250 * cur: cur = 0
    if cur > best: best, where = cur, "in-order"
    return best, where

def cert_prompt(it, p, k):
    return CERT.format(seq=", ".join(it["x_pref"]), k=k,
                       rule=(f"Rule: {p}\n" if p else ""))

def _score(it, text, ntok, hit, k, is_listing):
    """Turn one generation into a result. p_lam states the terms, so finding
    them in its output is expected and it is not put to the scoring judge."""
    n_ok, where = extract(text, it["x_post"][:k])
    res = dict(n_ok=n_ok, where=where, tokens=ntok, no_answer=hit, raw=text)
    if not is_listing:
        res = verify_scoring(it, res, k)
    return res

class StopWhenAnswered(StoppingCriteria):
    """Stop when the answer is there. What counts as there depends on the
    machine, so `allow_extract` is passed in rather than assumed.

    A machine that closes its reasoning has an answer after the marker, and its
    working is not an answer: stopping on terms found there cut generations off
    mid-thought, once at 64 tokens because the machine was restating the
    instruction. A machine that never closes its reasoning has nothing after a
    marker to read, so the terms in its working are the only answer there is,
    and refusing to stop on them costs the full budget every time."""
    def __init__(self, tok, n_prompt, x_post, k, allow_extract, every=32, min_new=48):
        self.tok, self.n0, self.x_post, self.k = tok, n_prompt, x_post, k
        self.allow_extract = allow_extract
        self.every, self.min_new, self.i = every, min_new, 0
    def __call__(self, input_ids, scores, **kw):
        self.i += 1
        if self.i < self.min_new or self.i % self.every: return False
        txt = self.tok.decode(input_ids[0][self.n0:], skip_special_tokens=True)
        # Whether the terms in the working count as an answer is a property of
        # the generation, not of the machine. Qwen3 usually closes its reasoning
        # and sometimes does not: on one item it never closed the tag, so there
        # was nothing after a marker to read, the criterion could not fire, and
        # establishing that the machine needed no help cost 10,500 tokens across
        # three rungs. When no marker has appeared, the working is all there is.
        if self.allow_extract or after_think(txt) == "":
            n_ok, _ = extract(txt, self.x_post)
            if n_ok >= self.k: return True
        # A finished answer line also stops it, right or wrong. The marker alone
        # is not enough: the machine echoes the instruction while restating the
        # task, and "ANSWER: t1, t2, ... (all 5 terms, nothing else)" splits into
        # enough fields to look finished, so the line has to hold as many items
        # as were asked for.
        post = after_think(txt)
        j = post.upper().rfind("ANSWER:")
        if j < 0: return False
        tail = post[j + 7:]
        if "\n" not in tail: return False
        items = [x for x in re.split(r"[,;]", tail.split("\n")[0]) if x.strip()]
        return len(items) >= self.k

def seed_for(*parts):
    """Seed immediately before a call, derived from what is being measured.
    Seeding once per run would leave every generation dependent on how many came
    before it, so a resumed run would land elsewhere. Derived this way a single
    program reproduces from the record alone, which is what makes the rerun in
    `_run_once` the same trajectory carried further rather than a fresh draw."""
    h = hashlib.md5("|".join(str(x) for x in parts).encode()).hexdigest()[:8]
    s = int(h, 16)
    torch.manual_seed(s)
    if DEVICE == "cuda": torch.cuda.manual_seed_all(s)
    return s

@torch.no_grad()
def generate_for(x_post, k, prompt, max_new, seed_parts=(), allow_extract=None):
    """(text, tokens_used, hit_budget). tokens_used is the work up to the point
    the answer appeared, so it is usable as a proxy for computation."""
    if allow_extract is None: allow_extract = STOP_ON_EXTRACT
    tok.padding_side = "left"
    msgs = [{"role": "user", "content": prompt}]
    try:
        ch = tok.apply_chat_template(msgs, tokenize=False,
                                     add_generation_prompt=True, **TEMPLATE_KW)
    except TypeError:
        # not every chat template takes the thinking argument
        ch = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = tok(ch, return_tensors="pt").to(DEVICE)
    n0 = enc.input_ids.shape[1]
    stop = StoppingCriteriaList([StopWhenAnswered(tok, n0, x_post, k, allow_extract)])
    if STOCHASTIC: seed_for(*seed_parts, max_new)
    g = model.generate(**enc, **GEN_KW, max_new_tokens=max_new,
                       eos_token_id=(EOS_IDS if EOS_IDS else tok.eos_token_id),
                       pad_token_id=tok.eos_token_id, stopping_criteria=stop)
    used = g.shape[1] - n0
    return tok.decode(g[0][n0:], skip_special_tokens=True), used, (used >= max_new - 2)


## 7 · Extraction and certification

The machine is asked to think and then put the answer after `ANSWER:`.
It does
not always comply, so the terms are looked for in three places: the answer line,
any line of the working that runs the sequence, and the terms appearing in order
anywhere in the text.
That last route catches numbered working.

A truncated generation is retried once at a larger budget, because a rule makes
the machine more verbose and truncation would otherwise count against exactly
the condition being tested.


In [ ]:
_judge_on = USE_JUDGE and bool(os.environ.get("GOOGLE_API_KEY"))
if USE_JUDGE and not _judge_on:
    log("  judges disabled: no GOOGLE_API_KEY")
if _judge_on:
    from google import genai
    _JC = genai.Client()
JUDGE_CALLS = {"n": 0, "grant": 0, "deny": 0, "error": 0, "verify": 0, "corrected": 0}

def _text_of(r):
    """Thinking models often return the answer in parts rather than in .text."""
    if getattr(r, "text", None): return r.text
    out = []
    for c in (getattr(r, "candidates", None) or []):
        for p in (getattr(getattr(c, "content", None), "parts", None) or []):
            t = getattr(p, "text", None)
            if t and not getattr(p, "thought", False): out.append(t)
    return "".join(out)

def _loads(txt):
    """Close a truncated object before parsing. The output budget covers the
    thinking as well, so a reply can stop in the middle of a field."""
    t = re.sub(r"```(?:json)?", "", txt or "")
    i = t.find("{")
    if i < 0: raise ValueError("no JSON in reply")
    frag = t[i:]
    for a in (frag, frag + '"}', frag + "}", frag + '""}', frag + 'false}', frag + '": false}'):
        try: return json.loads(a)
        except Exception: pass
    raise ValueError("unparseable: " + frag[:80])

JUDGE_PROMPT = """A small language model was asked to continue a sequence and ran
out of room before finishing. Decide whether giving it more room would help.

Sequence shown to it: {x_pref}
The correct next terms are: {x_post}
{rule}
Here is the end of what it wrote before it was cut off:
---
{tail}
---

Was it executing a procedure and making real progress toward those terms, or was
it guessing, going in circles, or working on something that will not reach them?
Derivations that correctly reproduce terms one by one are progress. Restating the
sequence it was given, trying unrelated patterns, or repeating the same failed
idea are not.

Answer in this order, so that the verdict follows the reasoning rather than
preceding it. Reply with strict JSON only:
{{"reason": "<at most 15 words>", "terms_derived": <how many correct terms it
had worked out>, "progressing": true or false}}"""

def judge_extend(it, p, raw):
    if not _judge_on: return False, None
    try:
        q = JUDGE_PROMPT.format(
            x_pref=", ".join(it["x_pref"][:20]), x_post=", ".join(it["x_post"][:8]),
            rule=(f"It was given this rule: {p}\n" if p else "It was given no rule.\n"),
            tail=raw[-2500:])
        r = _JC.models.generate_content(model=BUDGET_JUDGE, contents=q,
                config={"max_output_tokens": 8000, "temperature": 0.0,
                        "response_mime_type": "application/json"})
        d = _loads(_text_of(r))
        JUDGE_CALLS["n"] += 1
        g = bool(d.get("progressing"))
        JUDGE_CALLS["grant" if g else "deny"] += 1
        return g, d
    except Exception as e:
        JUDGE_CALLS["error"] += 1
        JUDGE_CALLS["last_error"] = str(e)[:120]
        return False, {"error": str(e)[:80]}

VERIFY_PROMPT = """A model was asked to continue a sequence.
The correct next terms are: {x_post}

Here is everything it wrote:
---
{raw}
---
Did it actually produce those terms, in order, as its answer or in its working?
Terms that only appear incidentally inside ordinary prose do not count, and
neither does the model restating the terms it was given.

Reply with strict JSON only:
{{"produced": <how many of the terms, in order, it really produced>}}"""

RESCUE_PROMPT = """A model was asked to continue a sequence.
The correct next terms are: {x_post}

Here is everything it wrote:
---
{raw}
---
Count how many of those terms, in order from the first, it actually worked out.
Working shown step by step counts, whether or not it wrote a final answer line.
Restating the terms it was given does not count, and neither does a term
appearing incidentally in ordinary prose.

Reply with strict JSON only:
{{"produced": <count>}}"""

def rescue_count(it, raw, k):
    """The parser needs terms delimited and close together. On a sequence of
    single characters worked out across a long derivation it saw 1 of 9 where
    the judge saw 8. This asks for the count directly, and the answer is
    recorded as coming from the judge."""
    if not (_judge_on and RESCUE): return None
    try:
        r = _JC.models.generate_content(model=SCORING_JUDGE,
            contents=RESCUE_PROMPT.format(x_post=", ".join(it["x_post"][:k]),
                                          raw=raw[-6000:]),
            config={"max_output_tokens": 8000, "temperature": 0.0,
                    "response_mime_type": "application/json"})
        n = int(_loads(_text_of(r))["produced"])
        JUDGE_CALLS["rescue"] = JUDGE_CALLS.get("rescue", 0) + 1
        return max(0, min(n, k))
    except Exception:
        JUDGE_CALLS["error"] += 1
        return None

TAU_PROMPT = """A model was asked to continue a sequence.
The correct next terms are: {x_post}

Here is everything it wrote:
---
{raw}
---
Quote the shortest exact phrase from its text that ENDS at the point where it
had produced all of those terms for the first time. Then say whether it later
revised or contradicted that answer.

Reply with strict JSON only:
{{"phrase": "<exact quote, at most 60 characters, ending at that point>",
  "revised_after": true or false}}"""

def tau_judged(raw, spent, x_post):
    """Tau for a generation that ran to its budget.

    When a generation stops on its own, the tokens it used are the work the
    program needed. When it fills its budget, that number is the budget and
    measures nothing, but the answer is still in the text. So the judge is asked
    where the answer was complete, and the prefix up to that point is tokenised
    exactly: no interpolation from characters is needed.

    It is also asked whether the machine changed its mind afterwards. If it did,
    the first complete enumeration was not the answer and tau stays at the end.
    OLMo wrote "f, g, h, i, j" and then went "wait, that is only 21 terms if we
    start from a, let me count again"."""
    if not (_judge_on and raw): return None, "no judge"
    try:
        r = _JC.models.generate_content(model=SCORING_JUDGE,
            contents=TAU_PROMPT.format(x_post=", ".join(x_post),
                                       raw=raw[-TAU_MAX_CHARS:]),
            config={"max_output_tokens": 8000, "temperature": 0.0,
                    "response_mime_type": "application/json"})
        d = _loads(_text_of(r))
    except Exception as e:
        JUDGE_CALLS["error"] += 1
        return None, f"judge failed: {str(e)[:40]}"
    JUDGE_CALLS["tau"] = JUDGE_CALLS.get("tau", 0) + 1
    if d.get("revised_after"):
        return spent, "revised its answer, so tau stays at the end"
    ph = (d.get("phrase") or "").strip()
    if not ph: return None, "no phrase returned"
    # searched in the whole text, not the slice the judge saw, so a phrase
    # quoted from the tail still resolves to its true position
    pos = raw.find(ph)
    if pos < 0:
        ph = ph[:30]                    # judges paraphrase, so try a shorter prefix
        pos = raw.find(ph)
    if pos < 0: return None, "phrase not found in the text"
    t = len(tok(raw[:pos + len(ph)], add_special_tokens=False).input_ids)
    return min(t, spent), "judged"

def verify_scoring(it, res, k_asked):
    """Only the substring route is checked; the answer-line and working-line
    routes parse delimited lists and do not have the problem."""
    if not _judge_on or res.get("where") != "in-order" or res["n_ok"] == 0:
        return res
    try:
        r = _JC.models.generate_content(model=SCORING_JUDGE,
            contents=VERIFY_PROMPT.format(x_post=", ".join(it["x_post"][:k_asked]),
                                          raw=res["raw"][-4000:]),
            config={"max_output_tokens": 8000, "temperature": 0.0,
                    "response_mime_type": "application/json"})
        n = int(_loads(_text_of(r))["produced"])
        JUDGE_CALLS["verify"] += 1
        if n != res["n_ok"]:
            JUDGE_CALLS["corrected"] += 1
            res["n_ok_substring"], res["n_ok"] = res["n_ok"], n
            res["scoring_verified"] = True
    except Exception as e:
        res["verify_error"] = str(e)[:60]
    return res

if _judge_on:
    _t = judge_extend({"x_pref": ["a", "b"], "x_post": ["c", "d"]}, "",
                      "computing term one, then term two")
    log(f"  judge test call: {'ok' if _t[1] and not _t[1].get('error') else 'FAILED'}"
        f"  {str(_t[1])[:90]}")

# ------------------------------------------------------------ EXTRACTION ---
def _run_once(it, p, k, max_new):
    """One generation, with a rerun when an early stop turned out to be wrong.

    Stopping on terms found in the working is right when the machine has
    enumerated the answer and wrong when it is listing them while reasoning
    about the problem. Nothing visible mid-generation separates those, and
    calling the judge every 32 tokens is not affordable. So stop early, let the
    judge look, and if it says the terms were not produced, run it again with
    the same seed and no early stop. The seed makes that the same trajectory
    carried further, so only false positives pay twice."""
    is_listing = (p == listing_of(it["x_post"]))
    prompt = cert_prompt(it, p, k)
    sp = (it["id"], p, MACHINE_ID)
    text, ntok, hit = generate_for(it["x_post"][:k], k, prompt, max_new, sp)
    res = _score(it, text, ntok, hit, k, is_listing)

    # A rerun is worth it whenever an early stop came from the terms in the
    # working and the judge then rejected them. That can happen on any machine,
    # since the fallback above fires whenever a generation has no marker.
    stopped_on_working = STOP_ON_EXTRACT or after_think(text) == ""
    if (stopped_on_working and res.get("scoring_verified") and res["n_ok"] < k
            and not res["no_answer"]):
        t2, n2, h2 = generate_for(it["x_post"][:k], k, prompt, max_new, sp,
                                  allow_extract=False)
        res2 = _score(it, t2, n2, h2, k, is_listing)
        res2["rerun_after_false_stop"] = True
        res2["tokens_first_try"] = ntok
        res = res2
    return res

def certify(it, p, budgets=None):
    """Certify p by generation, climbing a ladder of budgets.

    A run that fills its budget is put to the judge. The first rung is granted
    on 'progressing'; later rungs need the judge's count of terms derived to
    have risen since the last attempt, so a machine circling the same thought is
    dropped while one grinding through a derivation is carried."""
    x_post, k = it["x_post"], it["k"]
    # p_lam states the terms in the prompt, so the rescue below must not run on
    # it: asking a judge to count terms in a reply that was handed them is not a
    # check on anything. `_run_once` works this out for its own scoring and the
    # name was being read here as though it leaked out of that call, which it
    # does not. It cost the run at the second item, on the first program that
    # did not certify outright.
    is_listing = (p == listing_of(it["x_post"]))
    ladder = budgets or BUDGETS
    prev_derived, rungs, jv, rerun = -1, [], None, False
    for step, budget in enumerate(ladder):
        # _run_once handles the early stop, the substring check, and the rerun
        # when an early stop turns out to have been a false positive
        # Each rung is a full generation and the top one can be minutes. Say so
        # before it starts, and say how long it took, or a slow item is
        # indistinguishable from a stalled one.
        if step:
            log(f"      rung {step+1}/{len(ladder)}, {budget} tokens ...")
        _rt = time.time()
        res = _run_once(it, p, k, budget)
        if step:
            log(f"      rung {step+1} done in {time.time()-_rt:.0f}s, "
                f"{res['n_ok']}/{k} terms")
        text, ntok, hit = res["raw"], res["tokens"], res["no_answer"]
        n_ok, where = res["n_ok"], res["where"]
        sv, ns = res.get("scoring_verified"), res.get("n_ok_substring")
        rerun = rerun or bool(res.get("rerun_after_false_stop"))
        rungs.append(dict(budget=budget, tokens=ntok, n_ok=n_ok, hit=hit,
                          rerun=bool(res.get("rerun_after_false_stop"))))

        if n_ok >= k: break                # certified
        if not hit: break                  # it finished and was wrong
        if step == len(ladder) - 1: break  # no rung left

        grant, jv = judge_extend(it, p, text)
        if not grant: break
        derived = jv.get("terms_derived")
        derived = derived if isinstance(derived, int) else 0
        if step >= 1 and derived <= prev_derived:
            if EXTEND_ON_COMPLETE and derived >= k:
                log(f"      the judge saw all {k} terms derived and no more "
                    f"progress to make, so the next rung is for writing the "
                    f"answer, not for deriving it")
            else:
                jv = dict(jv, reason="no further terms than last time: "
                                     + str(jv.get("reason", "")))
                break                      # going in circles
        prev_derived = derived

    # the parser wants terms delimited and close together, which a derivation
    # spread over thousands of tokens is not. Ask the judge to count instead.
    rescued = None
    if n_ok < k and hit and not is_listing:
        rc = rescue_count(it, text, k)
        if rc is not None and rc > n_ok:
            rescued, n_ok = n_ok, rc

    # A generation that certified while filling its budget has a tau that is
    # the budget. The answer is in the text, so ask where it was complete.
    tau_j = tau_how = None
    if n_ok >= k and hit:
        tau_j, tau_how = tau_judged(text, rungs[-1]["tokens"], x_post[:k])

    out = dict(passed=n_ok >= k, n_ok=n_ok, where=where, k_asked=k,
               # tau is the work the final run needed. Earlier rungs are failed
               # attempts under budgets that were too small: GPU spent, not work
               # required. Both are kept because they answer different questions.
               tau=rungs[-1]["tokens"], tokens_spent=sum(r["tokens"] for r in rungs),
               tau_judged=tau_j, tau_judged_how=tau_how,
               tokens=rungs[-1]["tokens"], budget=rungs[-1]["budget"],
               rungs=rungs, stochastic=STOCHASTIC, no_answer=hit,
               rerun_after_false_stop=rerun,
               searching=(n_ok == 0 and hit), partial=(n_ok > 0 and hit),
               extended=len(rungs) > 1, judge=jv, cert_build=BUILD, raw=text)
    if sv: out["scoring_verified"], out["n_ok_substring"] = True, ns
    if rescued is not None:
        out["n_ok_parser"], out["n_ok_rescued"] = rescued, True
    return out

# What a field carries when it has nothing to say. This matters more than it
# looks: hints are certified cheapest first, and "none" is four characters, so
# it prices below every real hint and would take the first slot. A wasted
# certification is a couple of minutes of GPU.
HINT_JUNK = {"none", "n/a", "na", "nil", "null", "unknown", "unspecified",
             "-", "--", "?", "other", "misc", "general", "various", "any",
             "no domain", "not applicable", "tbd", "n.a."}

def _clean_hint(s):
    s = " ".join(str(s or "").split())
    if not s or s.lower().strip(". ") in HINT_JUNK:
        return None
    if not any(ch.isalpha() for ch in s):
        return None                       # digits or punctuation name nothing
    return s if 2 <= len(s) <= HINT_MAX_CHARS else None

def free_hints(it):
    """Hints already in the battery. No calls, no waiting.

    Ordered roughly by how much they give away, though the ORDER THAT MATTERS
    is by cost in bits, which is computed later and may differ.
    """
    out, seen = [], set()
    def add(tag, s):
        s = _clean_hint(s)
        if s and s.lower() not in seen:
            seen.add(s.lower()); out.append(dict(tag=tag, p=s))
    add("hint-theme", it.get("theme"))
    add("hint-domain", it.get("domain"))
    # the knowledge spans are exactly the part a prefix cannot reveal: which
    # table, which ordering. This is the hint most likely to be both short and
    # sufficient.
    know = [c.get("text") for c in (it.get("components") or [])
            if isinstance(c, dict) and c.get("type") == "knowledge"]
    for k in know[:3]:
        add("hint-knowledge", k)
    if len(know) > 1:
        add("hint-knowledge", "; ".join(str(k) for k in know[:3]))
    add("hint-mechanism", it.get("mechanism"))
    add("hint-gloss", it.get("gloss"))
    return out

HINT_PROMPT = """Here are the first {n} terms of a sequence:
{prefix}

It continues:
{post}

A reader will be shown ONLY the {n} terms above, plus one short note from you,
and asked to write the continuation. They cannot see this message.

Write the SHORTEST notes that would be enough. The prefix already shows the
shape: the arithmetic, the repetition, the index, the layout. What it usually
cannot show is WHICH remembered list or table is in play, and that is what a
note is for.

  terms  odod1, erer2, imim3, afaf4, loslos5
  note   solfege syllables, each doubled

  terms  8 bpm, 14 bpm, 21 bpm, 30 bpm
  note   letters in the element names, running total

Give {k} notes, shortest first, none longer than {maxc} characters. Do not
restate the terms. Do not write the whole rule if a name will do.
{feedback}
Answer in strict JSON, nothing else:
{{"hints": ["<shortest>", "..."]}}"""

HINT_RETRY = """
Your earlier notes were tried and did not work. What happened:
{tried}

A note fails when the reader still cannot tell what to do, or when it names
something the reader gets wrong. Write {k} different notes. They may be a
little longer if that is what it takes, but they must actually be enough."""

def asked_hints(it, feedback=""):
    """Hints proposed by the composer. One call."""
    if not HINTS_ASKED or not _judge_on:
        return []
    try:
        q = HINT_PROMPT.format(
            n=it["n"], prefix=", ".join(it["x_pref"]),
            post=", ".join(it["x_post"][:it["k"]]),
            k=HINT_ASK_N, maxc=HINT_MAX_CHARS, feedback=feedback)
        r = _JC.models.generate_content(
            model=SCORING_JUDGE, contents=q,
            config={"max_output_tokens": 4000, "temperature": 0.3,
                    "response_mime_type": "application/json"})
        d = _loads(_text_of(r))
        JUDGE_CALLS["n"] += 1
    except Exception as e:
        JUDGE_CALLS["error"] += 1
        JUDGE_CALLS["last_error"] = str(e)[:120]
        return []
    out, seen = [], set()
    for h in (d or {}).get("hints", []):
        s = _clean_hint(h)
        if s and s.lower() not in seen:
            seen.add(s.lower()); out.append(dict(tag="hint-asked", p=s))
    return out

GAP_PROMPT = """A reference machine was shown the first {n} terms of a sequence
and asked to continue it. It could not, so it was offered short hints. These
were too vague and it still failed:

{failed}

It succeeded only when given the rule in full:

  {rule}

The full rule is far more expensive to state than any of the hints, so the
difficulty we record for this item is an upper bound and probably a loose one.
What is missing is something in between: more informative than the hints above,
much shorter than the rule.

Write {k} of them. Each should name the MECHANISM without spelling out the
arithmetic: which tables are involved, in what order they are read, what is
carried from one term to the next. A reader who knows the domain should be able
to reconstruct the rule from it; a reader who does not should still be stuck.

Under {maxc} characters each, plain text, no quotes around them.

Answer in strict JSON, nothing else: {{"hints": ["...", "..."]}}"""

def gap_hints(it, failed_hints, rule, k=6):
    """More hints, aimed at the gap between the hints that failed and the rule
    that worked.

    On the v17 battery two items certified only on the full rule while every
    hint failed, and the gap was 82 and 107 bits: v17-0004 wanted 225 bits when
    its dearest hint cost 119. Every strong examinee answers that item
    perfectly, so 225 is a measurement of the search rather than of the item.

    The composer cannot price what it writes, so it is asked for MORE of the
    middle ground and the pricing decides which land in the band.
    """
    if not (HINTS_ASKED and _judge_on):
        return []
    try:
        q = GAP_PROMPT.format(
            n=it["n"], k=k, maxc=HINT_MAX_CHARS,
            failed="\n".join(f"  {h}" for h in failed_hints[:6]),
            rule=str(rule)[:400])
        r = _JC.models.generate_content(
            model=SCORING_JUDGE, contents=q,
            config={"max_output_tokens": 4000, "temperature": 0.5,
                    "response_mime_type": "application/json"})
        d = _loads(_text_of(r))
        JUDGE_CALLS["n"] += 1
    except Exception as e:
        JUDGE_CALLS["error"] += 1
        JUDGE_CALLS["last_error"] = str(e)[:120]
        return []
    out, seen = [], set()
    for h in (d or {}).get("hints", []):
        s = _clean_hint(h)
        if s and s.lower() not in seen:
            seen.add(s.lower()); out.append(dict(tag="hint-gap", p=s))
    return out

def pick_in_band(cands, lo, hi, k=3):
    """The k candidates spread most evenly across (lo, hi).

    Spread rather than cheapest: the cheapest is nearest the hints that already
    failed, so a run of cheap ones would repeat what is known. Taking them
    apart samples the band instead.
    """
    inband = sorted((c for c in cands if c.get("cost") is not None
                     and lo < c["cost"] < hi), key=lambda c: c["cost"])
    if len(inband) <= k:
        return inband
    step = (len(inband) - 1) / (k - 1) if k > 1 else 0
    return [inband[round(j * step)] for j in range(k)]

HINTX_PROMPT = """You wrote the rule below, and a reader must continue the
sequence from the prefix WITHOUT seeing it.

{rule}
Prefix, the {n} terms they are shown: {prefix}
What they must produce next: {post}

Write {k} HINTS. A hint is not the rule. It is the shortest thing you could
say that lets a reader who is good at arithmetic, and who knows the usual
tables, do what the rule does. You know the answer; the hint is what you would
whisper.

  rule   the n-th solfege syllable, doubled, then n
  hint   solfege syllables, each doubled

  rule   running total of the letter counts of the element names
  hint   letters in the element names, running total

SAY WHAT A TERM LOOKS LIKE, or tell the reader to follow the format of the
sequence, especially when terms are made of several elements. A hint about the
arithmetic alone tends to be answered with the arithmetic alone.

Name the tables so that only one reading is possible. A name that admits two
orderings will be argued over rather than used.

Name the mechanism if the prefix cannot show it. Leave out anything the prefix
already shows AND that your hint does not contradict. Shortest first, none
longer than {maxc} characters.
{feedback}
Answer in strict JSON, nothing else:
{{"hints": ["<shortest>", "..."]}}"""

HINTX_FEEDBACK = """
These hints were tried on the reference machine and did not work:
{tried}

A hint fails in one of two ways, and the counts above say which. Zero terms
right means it points at the wrong thing entirely. Some terms right means it
names part of what happens and not the rest; say what the hint left out.
Write {k} DIFFERENT hints."""

def _hintx_screen(it, hint):
    """Gemini continues the prefix from the hint alone. A sound negative test.

    Gemini is stronger than either reference machine, so it failing means the
    reference machine fails too; it succeeding says little. What that removes
    is hints that are WRONG or underspecified, which is the class worth
    removing: a hint naming Tarot on an item about pig materials cannot be
    saved by giving the reference machine more patience.

    A SOUNDNESS CHECK AND NOT A DIFFICULTY ONE. It asks for the first few terms
    only. Gemini is stronger than either reference machine, so whether it can
    carry twenty terms says nothing about whether the machine can; what it can
    settle is whether the hint points at the RIGHT THING at all. Five terms
    separate "Tarot Major Arcana" on an item about pig materials from "piece
    value times colour length times n", which is the rule.

    It was written to demand every one of the k_u terms on 2000 output tokens,
    which for twenty terms of `252 plies violet bishop` is not enough room to
    finish. It rejected five correct hints in a row on v17-0112, each costing
    thirty seconds, and the item never reached the machine.
    """
    if not HINTX_SCREEN or not _judge_on:
        return True, "screen off"
    _n = min(HINTX_SCREEN_TERMS, it["k"])
    want = it["x_post"][:_n]
    q = (f"Continue the sequence. You are given a note that describes how it "
         f"works.\n\nNote: {hint}\n\nThe sequence so far: "
         f"{', '.join(it['x_pref'])}\n\nGive exactly {_n} more terms, "
         f"comma separated, nothing else.")
    for _ in range(HINTX_SCREEN_REPS):
        try:
            r = _JC.models.generate_content(
                model=SCORING_JUDGE, contents=q,
                config={"max_output_tokens": HINTX_SCREEN_OUT,
                        "temperature": 0.3})
            JUDGE_CALLS["n"] += 1
            txt = _text_of(r)
        except Exception as e:
            JUDGE_CALLS["error"] += 1
            # PASSED THROUGH, not rejected. A screen that cannot run is not
            # evidence against the note, and silently dropping notes because
            # an API is down would look exactly like the note being bad.
            return True, f"screen errored, passing it through: {str(e)[:60]}"
        if not txt:
            continue
        got = [s.strip() for s in str(txt).replace("\n", ",").split(",")]
        got = [g for g in got if g][:_n]
        if got == want:
            return True, f"the first {_n} terms"
    return False, f"could not give the first {_n} terms"

def _hintx_rule(iid, S, it):
    """What the composer is shown: the description that CERTIFIED, where there
    is one, and the seed otherwise.

    It was shown the seed, which is the composer's own first prose and not
    necessarily anything the reference machine can run. The certified
    equivalent is: it is the one formulation known to work, so it is the right
    thing to compress. On v17-0112 the seed is prose about multiplying three
    things, while the equivalent binds them and says what a term looks like --
    `n x len(c) x v "plies" c p` -- which is exactly what the notes were
    failing to carry.

    Both are given only when they differ, and each is labelled, since a note
    compressing the wrong one of two unlabelled strings is worse than a note
    compressing one.
    """
    seed = it.get("rule") or ""
    p = (S.get("programs") or {}).get(S.get("p_star")) or {}
    won = p.get("p") if S.get("band") not in (None, "p_lam", "p_eps") else None
    if won and won.strip() and won.strip() != seed.strip():
        return (f"The rule as you first wrote it: {seed}\n"
                f"And a rewriting of it that WORKED, in the sense that the "
                f"reader was given this and produced the continuation "
                f"correctly: {won}\n"
                f"Your hints must say the same thing MORE BRIEFLY than that "
                f"one does, by leaving out whatever a reader could guess from "
                f"the prefix.")
    return f"Rule: {seed}"

def _hintx_ask(it, rule, tried, k, too_long=False, bar=None):
    """Hints from the composer, WITH the rule in front of it."""
    if not _judge_on:
        return []
    fb = ""
    if too_long:
        fb = ("\nThe hints you gave were CORRECT but too long to be worth "
              "using: a hint only helps if it is markedly shorter than the "
              "rewriting above. Say the same thing in far fewer words. Drop "
              "anything the prefix already shows, and prefer a name over a "
              "description wherever one exists.")
    elif tried:
        lines = "\n".join(f"  \"{h}\" -> {n} of {it['k']} terms right"
                          for h, n in tried)
        fb = HINTX_FEEDBACK.format(tried=lines, k=k)
    q = HINTX_PROMPT.format(rule=rule, n=it["n"],
                             prefix=", ".join(it["x_pref"]),
                             post=", ".join(it["x_post"][:it["k"]]),
                             k=k, maxc=HINTX_MAX_CHARS, feedback=fb)
    try:
        r = _JC.models.generate_content(
            model=SCORING_JUDGE, contents=q,
            config={"max_output_tokens": 4000, "temperature": 0.7,
                    "response_mime_type": "application/json"})
        JUDGE_CALLS["n"] += 1
        d = _loads(_text_of(r))
    except Exception as e:
        JUDGE_CALLS["error"] += 1
        JUDGE_CALLS["last_error"] = str(e)[:120]
        log(f"      the composer could not be asked: {str(e)[:80]}")
        return []
    out, seen = [], set()
    for h in (d or {}).get("hints", []):
        h = (str(h) or "").strip()
        if not h or len(h) > HINTX_MAX_CHARS: continue
        if h.lower() in HINT_JUNK or h.lower() in seen: continue
        seen.add(h.lower()); out.append(h)
    return out

def hintx_item(i, S, it, bar, goal):
    """One item. Returns the record written under S['hintx'].

    `bar` is the cost a hint must come in under to be worth running: the
    item's K_cond where it has one, or the listing's cost where it does not.
    `goal` is "lower" or "certify" and decides the order.

    DESCENDING in both cases, and for different reasons. Where nothing has
    certified the goal is EXISTENCE, so the longest note is the likeliest to
    work and the first success ends it. Where a K_cond exists the goal is the
    MINIMUM, so each success becomes the new bar and the next probe goes below
    it: a staircase, each step a real improvement, stopping when one fails or
    the budget of runs is spent.
    """
    rule = _hintx_rule(i, S, it)
    _too_long = False
    rec = dict(goal=goal, bar_at_start=round(bar, 2), rounds=[], best=None,
               ran=0, screened_out=0, asked=0, build=BUILD)
    tried = []
    for rnd in range(1, HINTX_ROUNDS + 1):
        hints = _hintx_ask(it, rule, tried, HINTX_ASK_N,
                           too_long=_too_long, bar=bar)
        rec["asked"] += len(hints)
        log(f"    round {rnd}: {len(hints)} hint(s) from the composer")
        if not hints:
            rec["rounds"].append(dict(round=rnd, asked=0)); continue

        # cost each, keep those under the bar, dearest first
        scored = []
        for h in hints:
            c = round(bits(h + "\n", F_rule_slot(it["x_pref"])), 2)
            scored.append((c, h))
        scored.sort()                    # cheapest first, for the search below
        under = [(c, h) for c, h in scored if c < bar]
        for c, h in scored:
            log(f"      {c:7.1f} bits  {'' if c < bar else 'OVER THE BAR  '}{h[:64]}")
        if not under:
            log(f"      none came in under {bar:.1f} bits, so none is worth a run")
            rec["rounds"].append(dict(round=rnd, asked=len(hints), under_bar=0))
            _too_long = True
            continue

        # IS THE BEST OF THEM WORTH A RUN AT ALL? A run costs twenty minutes to
        # an hour and a half, and a hint that would take a few per cent off the
        # bar is not worth that: the round is better spent asking for something
        # shorter. The next round is told so, and told by how much.
        _gain = (bar - under[0][0]) / bar if bar else 0.0
        if _gain < HINTX_MIN_GAIN and rnd < HINTX_ROUNDS:
            log(f"      the cheapest is {under[0][0]:.1f} against {bar:.1f}, "
                f"a saving of {_gain:.0%} and under the {HINTX_MIN_GAIN:.0%} "
                f"worth a run. Asking for shorter ones instead.")
            rec["rounds"].append(dict(round=rnd, asked=len(hints),
                                      under_bar=len(under), best_gain=round(_gain, 3),
                                      skipped="too close to the bar"))
            _too_long = True
            continue
        _too_long = False

        r_rows = []
        lo, hi = 0, len(under) - 1
        while lo <= hi and rec["ran"] < HINTX_RUN_N:
            mid = (lo + hi) // 2
            c, h = under[mid]
            if c >= bar:                 # a success has moved the bar below it
                hi = mid - 1
                log(f"      {c:.1f} is no longer under the bar {bar:.1f}, skipped")
                continue
            okscr, why = _hintx_screen(it, h)
            if not okscr:
                rec["screened_out"] += 1
                log(f"      {c:7.1f}  SCREENED OUT: {why}   {h[:50]}")
                r_rows.append(dict(cost=c, hint=h, screened=False, why=why))
                lo = mid + 1             # treat as a failure: look dearer
                continue
            log(f"      {c:7.1f}  screen ok, running on {MACHINE_ID}")
            _p = pid(h)
            _rr = S["programs"].setdefault(_p, dict(p=h, tag="hint-x"))
            _rr["c_p_given_xpref"] = c
            cert = certify(it, h)
            rec["ran"] += 1
            n_ok = (cert or {}).get("n_ok")
            passed = bool((cert or {}).get("passed"))
            _rr["cert_hintx"] = cert
            tried.append((h, n_ok if n_ok is not None else 0))
            r_rows.append(dict(cost=c, hint=h, screened=True,
                               passed=passed, n_ok=n_ok,
                               tau=(cert or {}).get("tau")))
            log(f"        -> {'CERTIFIED' if passed else 'no'}, "
                f"{n_ok} of {it['k']} terms, tau {(cert or {}).get('tau')}")
            if passed:
                rec["best"] = dict(hint=h, cost=c, tau=(cert or {}).get("tau"))
                bar = c
                if goal == "certify":
                    log(f"        the item is certified, which is the whole "
                        f"goal here; stopping"); break
                hi = mid - 1             # it works: is anything cheaper enough?
            else:
                lo = mid + 1             # too thin: look dearer
        if rec["ran"] >= HINTX_RUN_N:
            log(f"      {HINTX_RUN_N} runs spent, stopping")
        rec["rounds"].append(dict(round=rnd, asked=len(hints),
                                  under_bar=len(under), runs=r_rows))
        if rec["best"] and goal == "certify": break
        if rec["ran"] >= HINTX_RUN_N: break
    rec["bar_at_end"] = round(bar, 2)
    return rec

def hintx_run():
    """The whole pass. Writes only to S['hintx'] and to hint-x records.

    Nothing that any table reads is touched: K_cond, p_star, band and h keep
    the values they have. A separate step can fold the result in later, once
    there is a reason to believe it is worth folding.
    """
    stage(f"HINT EXTRA ROUND UNDER {MACHINE_ID}")
    MET = M.setdefault("items", {})
    # MEMBERSHIP IN ITEMS IS THE TEST. An item is there only if it is valid,
    # priced under this machine and certified by it, which is exactly the set
    # a rescue can apply to. Rebuilding that test here would be a second copy
    # of a rule that has been wrong before.
    want = []
    for i in ITEMS:
        if HINTX_ITEMS and i not in HINTX_ITEMS: continue
        S = (MET.get(i) or {}).setdefault(MACHINE_ID, {}) if i in MET \
            else MET.setdefault(i, {}).setdefault(MACHINE_ID, {})
        band = S.get("band")
        # p_eps ONLY is skipped: its K_cond is zero and nothing can go under
        # it. An item with no annotation at all is kept, because a note that
        # certifies it is worth as much there as anywhere, and more: it would
        # be the item's first program.
        if band == "p_eps":
            continue
        want.append((i, S, band))
    # the order asked for, when one was given
    if HINTX_ITEMS:
        want.sort(key=lambda w: HINTX_ITEMS.index(w[0]))

    log(f"  {len(want)} item(s) eligible under {MACHINE_ID}")
    log(f"  skipped: items with no annotation, and p_eps items, whose K_cond "
        f"is already zero and cannot be lowered")
    log(f"  {HINTX_ROUNDS} round(s), {HINTX_ASK_N} note(s) asked each, at "
        f"most {HINTX_RUN_N} run(s) on the reference machine per item")
    log(f"  the screen is {'ON: Gemini runs each hint first' if HINTX_SCREEN else 'OFF'}")
    log(f"  hints may run to {HINTX_MAX_CHARS} characters, against the "
        f"{HINT_MAX_CHARS} a hint is held to in the main pass")
    if not want:
        log("  nothing to do."); return

    lower = [w for w in want if w[2] != "p_lam"]
    certy = [w for w in want if w[2] == "p_lam"]
    log(f"\n  {len(lower)} with a K_cond to lower, {len(certy)} that fell to "
        f"the listing and would be rescued by any program at all")

    done = won = 0
    for i, S, band in want:
        it = ITEMS.get(i)
        if it is None:
            log(f"\n  {i}: could not be rebuilt from the battery, skipped"); continue
        goal = "lower" if (band and band != "p_lam") else "certify"
        bar = S.get("K_cond")
        if bar is None:
            # NO ANNOTATION, so no K_cond to go under. The bar is what the
            # listing costs: a note dearer than writing the terms out is not
            # worth running, and one cheaper would be this item's first
            # program. Computed here rather than read, since the annotation
            # that would have stored it has not run.
            bar = round(bits("".join(as_pieces(it["x_post"])),
                             F_next(it["x_pref"], "")), 2)
            log(f"    no annotation yet, so the bar is the listing at "
                f"{bar:.1f} bits")
        log(f"\n  [{done+1}/{len(want)}] {i}   band {band}, goal: "
            + ("certify it at all" if goal == "certify"
               else f"lower K_cond from {bar:.1f}"))
        log(f"    rule : {str(it.get('rule') or '')[:110]}")
        log(f"    shown {it['n']}, asked {it['k']}: "
            f"{', '.join(it['x_post'][:6])}")
        t0 = time.time()
        rec = hintx_item(i, S, it, bar, goal)
        rec["minutes"] = round((time.time() - t0) / 60, 1)
        S["hintx"] = rec
        done += 1
        if rec["best"]:
            won += 1
            log(f"    KEPT: {rec['best']['cost']:.1f} bits against "
                f"{rec['bar_at_start']:.1f}, a saving of "
                f"{rec['bar_at_start'] - rec['best']['cost']:.1f}   "
                f"[{rec['minutes']:.1f} min]")
        else:
            log(f"    nothing certified under the bar   "
                f"[{rec['minutes']:.1f} min, {rec['ran']} run(s), "
                f"{rec['screened_out']} screened out]")
        # save_and_maybe_push, not save() then state_save(): the first writes
        # only to the working directory and the second pushes every time. This
        # one respects STATE_PUSH_EVERY, which is what that setting is for.
        save_and_maybe_push(f"{BUILD}: extra hint round {done}/{len(want)}")

    log(f"\n  {won} of {done} items improved.")
    log(f"  Nothing in K_cond, p_star, band or h has been changed. The result "
        f"is under `rescue` on each machine slot, and a separate step folds it "
        f"in once there is a reason to.")
    _sav = [(i, (MET[i][MACHINE_ID]["hintx"]["bar_at_start"]
                 - MET[i][MACHINE_ID]["hintx"]["best"]["cost"]))
            for i, S, b in want
            if (MET.get(i, {}).get(MACHINE_ID, {}).get("hintx") or {}).get("best")]
    if _sav:
        log(f"  savings, bits: " + ", ".join(f"{i} {s:.0f}" for i, s in
                                             sorted(_sav, key=lambda z: -z[1])))
        log(f"  median {sorted(s for _, s in _sav)[len(_sav)//2]:.0f} bits over "
            f"{len(_sav)} item(s)")


## 8 · Run

For each item: price everything, then certify in ascending `c(p | x_pref)` with
an early stop at the first program that works, which is the minimum by
construction.
`p_eps` is tried first and `p_lam` last.

Progress is printed per program and state is saved after each one.


In [ ]:
# --------------------------------------------------------------- STORAGE ---
M = {"schema": 2, "build": BUILD, "machines": {}, "items": {}}
if os.path.exists(METRICS):
    M = json.load(open(METRICS))
    M.setdefault("machines", {}); M.setdefault("items", {})
    log(f"\nresuming from {METRICS}: {len(M['items'])} items on file")
CONDITIONS = dict(build=BUILD, k_max=K_MAX, top_n=TOP_N, budgets=BUDGETS,
                  decoding=("sampling" if STOCHASTIC else "greedy"), gen=GEN_KW,
                  think_end=THINK_END,
                  rescue=RESCUE, budget_judge=(BUDGET_JUDGE if USE_JUDGE else None),
                  scoring_judge=(SCORING_JUDGE if USE_JUDGE else None),
                  started=datetime.datetime.now().isoformat(timespec="seconds"))
_m = M["machines"].setdefault(MACHINE_ID, dict(**{k: v for k, v in
                              MACHINES[MACHINE_ID].items() if k != "gen"},
                              runs=[]))
_m.setdefault("runs", []).append(CONDITIONS)
log(f"conditions: k<={K_MAX}, budgets {BUDGETS}, {TOP_N} programs tried")
def save():
    json.dump(M, open(METRICS, "w"), indent=1)
    if OUTDIR != ".": shutil.copy(METRICS, OUTDIR)

# Pushing the whole state is a network upload, so it does not happen on
# every item. STATE_PUSH_EVERY handles a long run that might be cut off,
# and the summary cell pushes once at the end.
_pushed = {"n": 0}
def save_and_maybe_push(tag=""):
    """Write, and push every STATE_PUSH_EVERY calls.

    This existed and was NEVER CALLED. Every site called save() instead, which
    writes to /kaggle/working and no further, so STATE_PUSH_EVERY did nothing
    and a run reached the dataset only by running to the end. It cost two
    nights: a run stopped at item 7 pushed nothing, and the dataset kept a copy
    from before it started.
    """
    save()
    _pushed["n"] += 1
    if STATE_PUSH_EVERY and _pushed["n"] % STATE_PUSH_EVERY == 0:
        state_save(f"{BUILD}: {_pushed['n']} units done {tag}")
def slot(i): return M["items"].setdefault(i, {}).setdefault(MACHINE_ID, {})

_REDO = set(OLD) if REDO_OLD else set()
if _REDO:
    log(f"{len(_REDO)} control item(s) will be re-measured. Each is cleared "
        f"just before it is worked on, not all at the start, so stopping the "
        f"run costs\n  at most the item in progress. The previous band and h "
        f"are kept as band_prev and h_prev.")

def clear_for_redo(i):
    """Drop this item's certifications so it is measured afresh.

    Pricing is kept: it does not depend on the budgets. The previous band and
    h are kept beside the new ones so the two can be compared afterwards.
    """
    S = slot(i)
    old_band = S.get("band")
    for r in S.get("programs", {}).values():
        if "cert" in r: r["cert_prev"] = r.pop("cert")
    S["band_prev"], S["h_prev"] = old_band, S.get("h")
    S.pop("h", None); S.pop("band", None); S.pop("complete", None)
    log(f"   cleared: this item had band {old_band} and h "
        f"{S.get('h_prev')}, kept as band_prev and h_prev")

# Records made before the extract fixes are wrong, not merely measured under
# different budgets: lists introduced by prose scored zero, and the answer was
# looked for inside the reasoning. Clear them so they are redone.
STALE = ("Ctest-Annotator-v1.5", "Ctest-Annotator-v1.6", "Ctest-Annotator-v1.7")
_st = 0
for i in SELECTED:
    S = slot(i)
    if S.get("conditions", {}).get("build", "").startswith(STALE):
        for r in S.get("programs", {}).values(): r.pop("cert", None)
        S.pop("complete", None); _st += 1
if _st: log(f"cleared {_st} items certified under a build with the old extract; "
            f"pricing kept, so those cost generation only")
save()

todo = [i for i in SELECTED if not slot(i).get("complete")]
log(f"\n{len(todo)} items to annotate of {len(SELECTED)} selected, "
    f"{len(SELECTED)-len(todo)} already complete")
log(f"budget judge  {BUDGET_JUDGE if _judge_on else 'OFF, every failure is final at '
    + str(BUDGETS[0]) + ' tokens'}")
log(f"scoring judge {SCORING_JUDGE if _judge_on else 'OFF'}"
    + ("" if _judge_on else "  (set GOOGLE_API_KEY to turn them on)"))
log(f"hints         {'battery' if HINTS_FREE else 'off'}"
    + (" + composer" if HINTS_ASKED else "")
    + f", {HINT_TOP_N} certified per item at up to {HINT_BUDGETS[-1]} tokens")
log(f"              h minimises c(p|x_pref) over every string that certifies, "
    f"and a hint is one. Tried before the full programs, not instead of them.")
log("stop whenever: state is written after every program\n")

T0 = time.time()
_RUN_NOTE[0] = f"metrics for {MACHINE_ID}"
# THE RESCUE INSTEAD, when asked for. It reads what is already recorded
# and writes only to `rescue`, so it can be run long after the fact and
# undone by deleting one field.
# HINTX_ONLY SKIPS THE REST, it does not raise. SystemExit is how a notebook
# stops itself, but papermill turns it into a failed run: the pass finished and
# pushed, and the job still reported an error.
_SKIP_ANNOTATION = False
if HINTX_ONLY:
    hintx_run()
    _SKIP_ANNOTATION = True

for idx, i in enumerate([] if _SKIP_ANNOTATION else todo, 1):
    if i in _REDO:
        clear_for_redo(i)             # here, not at the start of the run
    it = ITEMS[i]; S = slot(i)
    _UNITS_SO_FAR[0] = idx - 1        # items FINISHED, not started

    log("=" * 72)
    _cell = it.get("cell") or (None, None)
    log(f"[{idx}/{len(todo)}] {i}   "
        + (f"r{_cell[0]}t{_cell[1]}   " if _cell[0] is not None else "")
        + f"n={it['n']}  k={it['k']}   {(time.time()-T0)/60:.0f} min elapsed")
    # What the pricing pass said about this item, so the annotation can be read
    # against it without opening another file. K is the description length the
    # unquestionability pass minimised to; h below is a different quantity,
    # conditional on the prefix and certified by this machine, and the two are
    # worth seeing side by side.
    _bits = []
    if it.get("K") is not None:
        _bits.append(f"K = {it['K']:.0f} bits")
    if it.get("slack") is not None:
        _bits.append(f"slack {it['slack']:+.0f}")
    _bits.append(f"n_M = {it['n']}")
    if it.get("tested") is False:
        _bits.append("certified against the structural rival alone")
    log(f"   from the pricing pass: " + ", ".join(_bits))
    def _show(v, first, last=1):
        if len(v) <= first + last:
            return ", ".join(v)
        return (", ".join(v[:first]) + f", ... , {v[-1]}")
    log(f"   x_pref ({len(it['x_pref'])} terms): {_show(it['x_pref'], 10)}")
    log(f"   x_post ({len(it['x_post'])} terms): {_show(it['x_post'], 8)}")
    S.setdefault("n", it["n"]); S.setdefault("k", it["k"]); S.setdefault("programs", {})
    S["conditions"] = CONDITIONS          # what THIS item was measured under

    if "c_xpref" not in S:
        S["c_xpref"] = round(bits(" " + ", ".join(it["x_pref"]), F_seq_alone()), 2)
    if "c_xpost_given_xpref" not in S:
        S["c_xpost_given_xpref"] = round(
            bits("".join(as_pieces(it["x_post"])), F_next(it["x_pref"], "")), 2)
    log(f"   c(x_pref) = {S['c_xpref']:.1f}   "
        f"c(x_post | x_pref) = {S['c_xpost_given_xpref']:.1f} bits")

    progs = programs_for(i)

    # Price and try p_eps first. If the machine continues the sequence unaided,
    # h is c(x_pref) and pricing the other fifty programs is work thrown away.
    _re = S["programs"].setdefault(pid(""), dict(p="", tag="p_eps"))
    if "c_p_given_xpref" not in _re:
        _re["c_p_given_xpref"] = round(bits("\n", F_rule_slot(it["x_pref"])), 2)
        _re["c_p"] = round(bits("\n", "Rule:"), 2)
        _re["c_xpref_given_p"] = S["c_xpref"]
        _re["c_xpost_given_p_xpref"] = S["c_xpost_given_xpref"]
    _re.update(cost=0.0, I=round(_re["c_p"] - _re["c_p_given_xpref"], 2),
               d=round(_re["c_p_given_xpref"] + _re["c_xpref_given_p"], 2), bayes=0.0)
    if "cert" not in _re:
        # Announced BEFORE the work. A certification climbs a ladder of budgets
        # and one rung at 6000 tokens can take five minutes on a T4, so without
        # this the log sits silent and there is no way to tell a slow item from
        # a hung one.
        log(f"   trying p_eps, the empty program, up to {BUDGETS[-1]} tokens ...")
        _t0 = time.time()
        _re["cert"] = certify(it, ""); save()
        log(f"   ({time.time()-_t0:.0f}s)")
    _c = _re["cert"]
    log(f"   {0.0:7.1f} bits  {_c['n_ok']:3d}/{it['k']:<3d} "
        f"{'CERTIFIED' if _c['passed'] else 'no':<10s} {_c.get('tau',_c['tokens']):5d}/"
        f"{_c.get('budget','?')} tok"
        f"{'' if _c['passed'] else (' SEARCHING' if _c.get('searching') else (' PARTIAL' if _c.get('partial') else ''))}"
        f"{'+' + '/'.join(str(r['budget']) for r in _c.get('rungs', [])[1:]) if _c.get('extended') else ''}"
        f"{' judged' if _c.get('n_ok_rescued') else ''}  (p_eps)")
    if _c.get("tau_judged") is not None:
        log(f"        tau judged {_c['tau_judged']} of the {_c['tau']} generated: "
            f"the answer was complete before the machine stopped writing")
    elif _c.get("tau_judged_how"):
        log(f"        tau not recovered: {_c['tau_judged_how']}")
    _jv = _c.get("judge") or {}
    if _jv and not _c["passed"]:
        log(f"        budget judge "
            + (f"raised the budget to {_c['budget']}, still did not finish"
               if _c.get("extended") else "refused more tokens")
            + f": {_jv.get('reason', _jv.get('error',''))[:90]}")

    if _c["passed"]:
        p_star = pid("")
        log(f"   the machine needs no help, so the other {len(progs)-1} programs "
            f"are not priced")
    else:
        p_star = None
        # Ask the composer for hints now, before pricing, so they are priced
        # with everything else and take part in the same ordering. One call,
        # and only on items where p_eps failed, which is where a hint could
        # lower h at all.
        if HINTS_ASKED and not S.get("asked_hints_done"):
            _t0 = time.time()
            _new = asked_hints(it)
            S["asked_hints_done"] = True
            _have = {c["p"] for c in progs}
            _new = [c for c in _new if c["p"] not in _have]
            progs = progs + _new
            if _new:
                log(f"   composer proposed {len(_new)} hint(s) in "
                    f"{time.time()-_t0:.0f}s: "
                    + "; ".join(repr(c["p"][:38]) for c in _new[:3]))
        _nh = sum(1 for c in progs if str(c["tag"]).startswith("hint"))
        log(f"   {len(progs)-1} other programs, pricing "
            f"({_nh} of them hints) ...")
    for c in ([] if _c["passed"] else progs):
        hid = pid(c["p"]); r = S["programs"].setdefault(hid, dict(p=c["p"], tag=c["tag"]))
        if "c_p_given_xpref" not in r:
            r["c_p_given_xpref"] = round(bits(" " + c["p"] + "\n" if c["p"] else "\n",
                                              F_rule_slot(it["x_pref"])), 2)
        if "c_p" not in r:
            r["c_p"] = round(bits(c["p"] + "\n" if c["p"] else "\n", "Rule:"), 2)
        if "c_xpref_given_p" not in r:
            r["c_xpref_given_p"] = round(bits(" " + ", ".join(it["x_pref"]),
                                    F_rule_first(c["p"]) if c["p"] else F_seq_alone()), 2)
        if "c_xpost_given_p_xpref" not in r:
            r["c_xpost_given_p_xpref"] = round(
                bits("".join(as_pieces(it["x_post"])), F_next(it["x_pref"], c["p"])), 2)
    base = S["programs"][pid("")]["c_p_given_xpref"]   # p_eps is zero by convention
    for r in S["programs"].values():
        r["cost"]  = round(r["c_p_given_xpref"] - base, 2)
        r["I"]     = round(r["c_p"] - r["c_p_given_xpref"], 2)
        r["d"]     = round(r["c_p_given_xpref"] + r["c_xpref_given_p"], 2)
        r["bayes"] = round(r["c_p_given_xpref"] + S["c_xpref"]
                           - r["c_p"] - r["c_xpref_given_p"], 2)
    save_and_maybe_push("programs")

    # ---- choosing what to certify --------------------------------------
    #
    # WITHIN the hints, cost and sufficiency run together: a more informative
    # hint is a longer one. So the hints are a threshold and binary search
    # finds it, which matters because the answer is usually in the middle. On a
    # sequence of w, c, s, f the hint "base 4" is too little and the full rule
    # at 103 bits is far more than needed; "base 4 with digits w, c, s, f" sits
    # between, and taking the three cheapest never reaches it.
    #
    # ACROSS the kinds it does not hold, and that is why the search is not one
    # sweep over everything sorted by cost. A 45-bit hint can work where a
    # 100.8-bit rule fails, because they say different kinds of thing. Searching
    # the merged list would probe the rule, see it fail, and discard every
    # cheaper candidate including the hint that works. Simulated on the last
    # run's numbers it returned 103 and missed the 45.
    #
    # So: binary search the hints, then the cheapest rules as before, then the
    # listing. The rules keep their three slots, since the cheapest rule is not
    # reliably the one that works.
    _sorted = sorted(S["programs"].items(), key=lambda kv: kv[1]["cost"])
    _hints = [h for h, r in _sorted if str(r["tag"]).startswith("hint")]
    _rules = [h for h, r in _sorted
              if r["tag"] not in ("p_eps", "p_lam")
              and not str(r["tag"]).startswith("hint")]
    _lam = pid(listing_of(it["x_post"]))

    def _run(hid, why=""):
        r = S["programs"][hid]
        if "cert" not in r:
            _is_hint = str(r["tag"]).startswith("hint")
            _lad = HINT_BUDGETS if _is_hint else BUDGETS
            log(f"   {why}{r['cost']:6.1f} bits  {r['tag']:14s} "
                f"{r['p'][:44]!r} (up to {_lad[-1]} tok) ...")
            _t0 = time.time()
            r["cert"] = certify(it, r["p"], budgets=_lad); save()
            log(f"   ({time.time()-_t0:.0f}s)")
        return r["cert"]["passed"]

    order = []
    if not _c["passed"]:
        _won = None
        if _hints:
            log(f"   {len(_hints)} hints from "
                f"{S['programs'][_hints[0]]['cost']:.0f} to "
                f"{S['programs'][_hints[-1]]['cost']:.0f} bits; binary search, "
                f"at most {HINT_PROBES} probes")
            lo, hi, pr = 0, len(_hints) - 1, 0
            while lo <= hi and pr < HINT_PROBES:
                mid = (lo + hi) // 2; pr += 1
                if _run(_hints[mid], "probe "):
                    _won = mid; hi = mid - 1      # look cheaper
                else:
                    lo = mid + 1                  # look dearer
            # cost and sufficiency agree only as a tendency, so check just under
            _b = 0
            while _won is not None and _won > 0 and _b < PROBE_BELOW:
                _b += 1
                if _run(_hints[_won - 1], "under "):
                    _won -= 1
                else:
                    break
            if _won is not None:
                log(f"   cheapest hint that works: "
                    f"{S['programs'][_hints[_won]]['cost']:.1f} bits after "
                    f"{pr + _b} probes")
            else:
                log(f"   no hint certified in {pr} probes; falling through to "
                    f"the stated rules")
        if _won is None:
            _bar = ((it.get("K") or 0) + DELTA
                    if (PRUNE_ABOVE_BAR and DELTA and it.get("K")) else None)
            _cands = [h for h in _rules
                      if _bar is None
                      or (S["programs"][h].get("cost") or 0) <= _bar]
            _skipped = len(_rules) - len(_cands)
            if _skipped:
                log(f"   {_skipped} candidate(s) cost more than K + delta = "
                    f"{_bar:.0f} bits and cannot be chosen; not run")
            _cands = _cands[:TOP_N] if TOP_N else _cands
            _best = None
            for hid in _cands:
                # nothing dearer than a program that already worked can lower
                # the minimum, so stop when the remaining candidates are all
                # dearer than the best so far
                if _best is not None and (S["programs"][hid].get("cost") or 0) >= _best:
                    log(f"   the rest cost more than the {_best:.1f} bits that "
                        f"already certified; stopping")
                    break
                if _run(hid, "rule  "):
                    _c = S["programs"][hid].get("cost")
                    if _best is None or _c < _best:
                        _best = _c
                    if not KEEP_GOING_AFTER_SUCCESS:
                        break
            if (CLOSE_EQ_HINT_GAP and _best is not None
                    and not S.get("gap_closed")):
                _hb = [S["programs"][h].get("cost") for h in _rules
                       if str(S["programs"][h].get("tag", "")).startswith("hint")
                       and S["programs"][h].get("cost") is not None]
                _hok = any((S["programs"][h].get("cert") or {}).get("passed")
                           for h in _rules
                           if str(S["programs"][h].get("tag","")).startswith("hint"))
                _gap = (_best - max(_hb)) if _hb else 0
                if _hb and not _hok and _gap > GAP_MIN_BITS:
                    S["gap_closed"] = True
                    log(f"   the rule cost {_best:.0f} bits and the dearest "
                        f"hint {max(_hb):.0f}, a gap of {_gap:.0f}. Asking for "
                        f"descriptions in between.")
                    _failed = [S["programs"][h].get("p") for h in _rules
                               if str(S["programs"][h].get("tag","")).startswith("hint")]
                    _win = next((S["programs"][h].get("p") for h in _rules
                                 if (S["programs"][h].get("cert") or {}).get("passed")), "")
                    _extra = gap_hints(it, _failed, _win, k=GAP_ASK_N)
                    if _extra:
                        _have = {S["programs"][h].get("p") for h in S["programs"]}
                        _extra = [c for c in _extra if c["p"] not in _have]
                        for c in _extra:
                            _h = pid(c["p"])
                            S["programs"][_h] = dict(c, cost=price(c["p"], it))
                        _band = pick_in_band(
                            [dict(S["programs"][pid(c["p"])], _id=pid(c["p"]))
                             for c in _extra],
                            max(_hb), _best, k=GAP_TRY_N)
                        log(f"   {len(_extra)} proposed, "
                            f"{sum(1 for c in _extra if max(_hb) < (S['programs'][pid(c['p'])].get('cost') or 0) < _best)} "
                            f"landed in the band; trying {len(_band)}")
                        for c in _band:
                            if _run(c["_id"], "gap   "):
                                _cc = S["programs"][c["_id"]].get("cost")
                                if _cc is not None and _cc < _best:
                                    _best = _cc
                    else:
                        log(f"   the composer proposed nothing usable")

            if _best is None:
                _run(_lam, "lam   ")
        order = [h for h, r in _sorted if "cert" in r] + \
                ([_lam] if "cert" in S["programs"].get(_lam, {}) else [])
        order = list(dict.fromkeys(order))

    for _oi, hid in enumerate(order, 1):
        r = S["programs"][hid]
        c = r["cert"]
        # p_lam states x_post outright, so passing it is a check on the machine
        # rather than evidence about the item. Say so.
        kind = ("CERTIFIED" if r["tag"] != "p_lam" else "COPIED") if c["passed"] else "no"
        # The machine always writes text, so the useful thing to say is what was
        # in it when the budget ran out.
        #   SEARCHING: still hunting for the pattern. The text is working out
        #              differences, trying candidate rules, restating the terms.
        #              None of the target terms had been produced.
        #   PARTIAL:   some of the target terms were there, so it had found
        #              something and was part way through writing it out.
        # only describe how it failed when it did fail
        flag = ("" if c["passed"] else
                " SEARCHING" if (c.get("searching") or c.get("stalled")) else
                " PARTIAL" if (c.get("partial") or c.get("cut_off")) else "")
        if c.get("extended"):
            flag += "+" + "/".join(str(r["budget"]) for r in c.get("rungs", [])[1:])
        if c.get("n_ok_rescued"): flag += " judged"
        # the screen asks for fewer terms than the full run, so score against
        # what was asked rather than against the full k
        denom = c.get("k_asked", it["k"])
        log(f"   {r['cost']:7.1f} bits  {c['n_ok']:3d}/{denom:<3d} {kind:<10s}"
            f" {c.get('tau', c['tokens']):5d}/{c.get('budget','?')} tok{flag:<9s}"
            f"  {r['p'][:46] or '(p_eps)'}")

        if c.get("tau_judged") is not None:
            log(f"        tau judged {c['tau_judged']} of the {c['tau']} generated: "
                f"the answer was complete before the machine stopped writing")
        elif c.get("tau_judged_how"):
            log(f"        tau not recovered: {c['tau_judged_how']}")
        jv = c.get("judge") or {}
        if jv and not c["passed"]:
            if jv.get("error"):
                log(f"        budget judge could not answer: {jv['error'][:80]}")
            else:
                what = (f"raised the budget to {c['budget']}, still did not finish"
                        if c.get("extended") else "refused more tokens")
                nd = jv.get("terms_derived")
                log(f"        budget judge {what}"
                    + (f", saw {nd} terms derived" if nd is not None else "")
                    + f": {jv.get('reason', '')[:90]}")
        if c["passed"]: p_star = hid; break

    if p_star is None:
        p_star = pid(listing_of(it["x_post"])); S["listing_failed"] = True
        log(f"     {S['programs'][p_star]['cost']:7.1f} bits    -/-   ASSUMED   "
            f"p_lam is correct by construction, but the machine did not reproduce\n"
            f"                              x_post from it, so h here is an upper bound "
            f"the machine could not confirm")

    ps = S["programs"][p_star]
    # TWO minimisers, and they are different objects that have shared a name.
    #
    #   K_uncond, p_star_uncond   argmin c(p) over the programs the executor
    #               verified, with no prefix in view. Written by the
    #               unquestionability pass.
    #   K_cond,   p_star_cond     argmin c(p | x_pref) over the programs THIS
    #               machine certified with the prefix in front of it. That is
    #               what h is built from, and it can be the empty program, a
    #               hint or the listing, so it is often a different string.
    #
    # Both are a K, a description length under M, which is why they are named
    # alike. h = K_cond + c(x_pref), matching h_M = K_M(x_next | x_shown).
    #
    # Both are recorded here, as text and not only as a hash, so a record can
    # be read without joining it back to the pricing file. The gap between them
    # is the interesting quantity: it is how much of a program the prefix pays
    # for.
    S["p_star"] = p_star                          # the hash, as before
    S["p_star_tag"] = ps["tag"]
    S["p_star_cond"] = ps["p"]                    # the text of the winner
    S["K_cond"] = ps["cost"]                      # min c(p | x_pref), certified
    S["c_pstar"] = ps["cost"]
    S["p_star_uncond"] = it.get("p_star_uncond")  # from the pricing pass
    S["K_uncond"] = it.get("K")
    if it.get("K") is not None:
        S["prefix_pays"] = round(it["K"] - ps["cost"], 2)
    # h = K_cond + c(x_pref), which is the plan's h_M = K_M(x_next | x_shown)
    S["h"] = round(ps["cost"] + S["c_xpref"], 2)
    S["band"] = ("p_eps" if ps["tag"] == "p_eps" else
                 "p_lam" if ps["tag"] == "p_lam" else "rule")

    # Could a larger budget change this verdict? Only if the winner was p_lam
    # (nothing cheaper worked) AND something ran out of room rather than being
    # simply wrong. Those items are worth revisiting; the rest are settled.
    tried = [r["cert"] for r in S["programs"].values() if "cert" in r]
    ran_out = [c for c in tried if c.get("searching") or c.get("partial")
               or c.get("stalled") or c.get("cut_off")]   # old names too
    S["budget_limited"] = bool(S["band"] == "p_lam" and ran_out)
    S["n_hit_budget"] = len(ran_out)
    S["complete"] = True
    if S.get("K_uncond") is not None:
        log(f"   K_cond {S['K_cond']:.1f} bits   vs   K_uncond "
            f"{S['K_uncond']:.1f} bits   the prefix pays for "
            f"{S['prefix_pays']:+.1f}")
    log(f"   -> band {S['band']}   h = K_cond {S['K_cond']:.1f}"
        f" + c(x_pref) {S['c_xpref']:.1f} = {S['h']:.1f} bits"
        + (f"   [budget limited: {S['n_hit_budget']} programs ran out of room]"
           if S["budget_limited"] else ""))
    save_and_maybe_push("items")
    rate = (time.time()-T0)/60/idx
    log(f"   saved. {rate:.1f} min/item, ~{rate*(len(todo)-idx):.0f} min left\n")


## 9 · Summary

Three things to check in the output.
Whether `p_lam` ever failed to certify,
which would mean the machine cannot copy and nothing below it is meaningful.
How many generations were truncated.
And the Bayes residual, which should be
small if the machine is close to coherent.


In [ ]:
# --------------------------------------------------------------- SUMMARY ---
log("=" * 72)
ann = [i for i in M["items"] if M["items"][i].get(MACHINE_ID, {}).get("complete")]
log(f"annotated with {MACHINE_ID}: {len(ann)} items")
log("bands: " + str(dict(collections.Counter(M["items"][i][MACHINE_ID]["band"] for i in ann))))
if ann:
    hs = sorted(M["items"][i][MACHINE_ID]["h"] for i in ann)
    log(f"h: median {hs[len(hs)//2]:.0f}  q1 {hs[len(hs)//4]:.0f}  q3 {hs[3*len(hs)//4]:.0f} bits")
    log(f"c(x_pref): median "
        f"{st.median(M['items'][i][MACHINE_ID]['c_xpref'] for i in ann):.0f} bits")
    # r.get, not r[...]: the extra hint round adds `hint-x` programs that
    # carry only what it needs, and a bare subscript raised KeyError here and
    # lost the whole summary AFTER the pass had finished and saved. Same shape
    # of fault as the one the note beside this cell already records.
    res = [r["bayes"] for i in ann
           for r in M["items"][i][MACHINE_ID]["programs"].values()
           if "bayes" in r]
    log(f"Bayes residual c(p|x_pref)+c(x_pref)-c(p)-c(x_pref|p): "
        f"median {st.median(res):+.1f} bits, 0 would mean a coherent machine")
    certs = [r["cert"] for i in ann
             for r in M["items"][i][MACHINE_ID]["programs"].values() if "cert" in r]
    if certs:
        log(f"certifications: {len(certs)}")
        log("  stopped at the screen: "
            f"{sum(1 for c in certs if c.get('stage')=='screen')}"
            f", reached the full run: {sum(1 for c in certs if c.get('stage')=='full')}")
        log(f"  SEARCHING, still hunting the pattern when the budget ran out: "
            f"{sum(1 for c in certs if c.get('searching') or c.get('stalled'))}")
        log(f"  PARTIAL, some terms written when the budget ran out: "
            f"{sum(1 for c in certs if c.get('partial') or c.get('cut_off'))}")
        tk = [c["tokens"] for c in certs if not c.get("no_answer")]
        if tk:
            log(f"  tokens when it finished: median {int(st.median(tk))}, max {max(tk)}")
        ok = [c["tokens"] for c in certs if c["passed"]]
        if ok: log(f"     of which, when the program worked: median {int(st.median(ok))}")
        builds = collections.Counter(c.get("cert_build", "pre-1.2.0") for c in certs)
        if len(builds) > 1:
            log(f"  certified under more than one build: {dict(builds)}")
            log("  token counts are only comparable within a build")
    ext = [c for c in certs if c.get("extended")]
    if ext:
        won = sum(1 for c in ext if c["passed"])
        log(f"  extra tokens allowed on {len(ext)} generations; {won} of them then "
            f"reproduced the terms")
        if won == 0:
            log("     none paid off, so the budget judge is granting too freely")
    if _judge_on and JUDGE_CALLS["n"]:
        log(f"  budget judge: {JUDGE_CALLS['n']} calls, {JUDGE_CALLS['grant']} granted, "
            f"{JUDGE_CALLS['deny']} denied, {JUDGE_CALLS['error']} failed")
    if JUDGE_CALLS.get("verify"):
        log(f"  scoring judge: {JUDGE_CALLS['verify']} substring hits checked, "
            f"{JUDGE_CALLS.get('corrected',0)} corrected")

    # the control items: did the new settings change their verdicts?
    ctrl = [i for i in ann if "band_prev" in M["items"][i][MACHINE_ID]]
    if ctrl:
        S = M["items"]
        same = [i for i in ctrl if S[i][MACHINE_ID]["band"] == S[i][MACHINE_ID]["band_prev"]]
        log(f"\ncontrols: {len(ctrl)} items redone under the new settings")
        log(f"  same band as before: {len(same)} of {len(ctrl)}")
        for i in ctrl:
            a, b = S[i][MACHINE_ID]["band_prev"], S[i][MACHINE_ID]["band"]
            ha, hb = S[i][MACHINE_ID].get("h_prev"), S[i][MACHINE_ID]["h"]
            mark = "  same" if a == b else "  MOVED"
            log(f"    {i:10s} {str(a):6s} -> {str(b):6s}  h {ha} -> {hb}{mark}")
        moved = [i for i in ctrl if S[i][MACHINE_ID]["band"] != S[i][MACHINE_ID]["band_prev"]]
        if moved:
            log("  items that moved out of p_lam were being limited by the old budget,")
            log("  not by the item. That is the effect the judges were added to catch.")
    bl = [i for i in ann if M["items"][i][MACHINE_ID].get("budget_limited")]
    log(f"\nbudget limited: {len(bl)} of {len(ann)} items")
    if bl:
        log("  these landed in p_lam with programs that ran out of room rather than")
        log("  being wrong, so a larger budget could still move them:")
        log("     " + ", ".join(sorted(bl)))
        log("  to revisit only these, later:")
        log("     REDO = [...]  # the list above")
        log("     for i in REDO:")
        log("         S = M['items'][i][MACHINE_ID]")
        log("         for r in S['programs'].values(): r.pop('cert', None)")
        log("         S.pop('complete', None)")
        log("     save()   # then raise the budgets and rerun the loop")
    settled = [i for i in ann if not M["items"][i][MACHINE_ID].get("budget_limited")]
    log(f"settled: {len(settled)} items, where the verdict will not move with more tokens")
    lf = sum(1 for i in ann if M["items"][i][MACHINE_ID].get("listing_failed"))
    if lf: log(f"!! p_lam failed to certify on {lf} items")
log(f"\nwrote {METRICS}" + (f", copied to {OUTDIR}" if OUTDIR != "." else ""))


## 10 · Against what these items were last seen as

The point of a stratified run.
Known-easy items should stay `p_eps`, known-rule
items should stay `rule`, and any known-hard item that leaves `p_lam` was being
held there by the old budget rather than by the item.
The two judge lines say
whether the judges paid for themselves.


In [ ]:
log("=" * 72)
log("AGAINST WHAT THESE ITEMS WERE LAST SEEN AS\n")

got = {}
for i in SELECTED:
    S = M["items"].get(i, {}).get(MACHINE_ID, {})
    if S.get("complete"): got[i] = S["band"]

by = {"p_eps": "known easy", "rule": "known rule", "p_lam": "known hard", None: "never seen"}
for exp in ("p_eps", "rule", "p_lam", None):
    grp = [i for i in SELECTED if EXPECTED[i] == exp and i in got]
    if not grp: continue
    same = [i for i in grp if got[i] == exp]
    log(f"{by[exp]:12s} {len(grp)} items"
        + (f", {len(same)} unchanged" if exp else ""))
    for i in grp:
        S = M["items"][i][MACHINE_ID]
        mark = ("" if exp is None else
                "  same" if got[i] == exp else
                "  MOVED  <-- worth reading" )
        log(f"   {i:10s} {str(exp) or '-':6s} -> {got[i]:6s}  "
            f"h={S['h']:7.1f}  {S.get('n_hit_budget',0)} programs ran out{mark}")
    log("")

# ---------------------------------------------------------------------------
# The two minimisers side by side. p*_uncond is what the unquestionability pass
# minimised with no prefix in view, and K is read off it. p*_cond is what this
# machine certified with the prefix in front of it, and h is built from it. The
# difference is what the prefix pays for, which is the quantity the hints were
# added to expose.
_pp = [M["items"][i][MACHINE_ID] for i in SELECTED if i in got]
_pp = [r for r in _pp if r.get("prefix_pays") is not None]
if _pp:
    _g = sorted(r["prefix_pays"] for r in _pp)
    log(f"what the prefix pays for, K - c(p*|x_pref), over {len(_g)} items:")
    log(f"   median {_g[len(_g)//2]:+.0f} bits, from {_g[0]:+.0f} to {_g[-1]:+.0f}")
    _neg = sum(1 for x in _g if x < 0)
    if _neg:
        log(f"   {_neg} negative: the certified program cost MORE with the "
            f"prefix in view than the cheapest program did without it. That is")
        log(f"   possible, because the two are minima over different sets and "
            f"the conditional one ranges only over what this machine could run.")
    _bb = {}
    for r in _pp: _bb.setdefault(r.get("band"), []).append(r["prefix_pays"])
    for b, v in sorted(_bb.items(), key=lambda kv: str(kv[0])):
        log(f"   band {str(b):6s} n={len(v):2d}  median {sorted(v)[len(v)//2]:+.0f}")
    _hint = [r for r in _pp if str(r.get("p_star_tag","")).startswith("hint")]
    if _hint:
        log(f"   {len(_hint)} of {len(_pp)} items were won by a HINT rather than "
            f"a full program")
        for r in _hint[:5]:
            log(f"      {r.get('K_cond', r.get('c_pstar', 0)):6.1f} bits  "
                f"{str(r.get('p_star_cond'))[:60]!r}")
log("")

# the judges, and whether they paid for themselves
certs = [r["cert"] for i in SELECTED if i in got
         for r in M["items"][i][MACHINE_ID]["programs"].values() if "cert" in r]
if certs:
    ext = [c for c in certs if c.get("extended")]
    log(f"budget judge: {len(ext)} generations given more room, "
        f"{sum(1 for c in ext if c['passed'])} of them then reproduced the terms")
    if ext and not any(c["passed"] for c in ext):
        log("   none paid off. Either the judge is too generous or the extra room")
        log("   is not what those programs were missing.")
    resc = [c for c in certs if c.get("n_ok_rescued")]
    if resc:
        log(f"judge counted the terms on {len(resc)} generations the parser could "
            f"not read, {sum(1 for c in resc if c['passed'])} of which then certified")
        for c in resc[:5]:
            log(f"      parser {c.get('n_ok_parser')} -> judge {c['n_ok']} of {c['k_asked']}")
    taus = [c["tau"] for c in certs if c.get("tau") and c["passed"]]
    if taus:
        log(f"tau on programs that certified: median {int(st.median(taus))} tokens, "
            f"min {min(taus)}, max {max(taus)}")
        spent = sum(c.get("tokens_spent", c["tokens"]) for c in certs)
        log(f"   generated {spent:,} tokens in total to establish that")
    if STOCHASTIC:
        log("   this machine samples, so tau is one draw rather than a fixed "
            "quantity; two draws on the same program differed by 20 percent")
    rungs = [len(c.get("rungs", [])) for c in certs if c.get("rungs")]
    if rungs:
        import collections as _co
        log(f"budget rungs used: {dict(_co.Counter(rungs))}")
    ver = [c for c in certs if c.get("scoring_verified")]
    log(f"scoring judge: {len(ver)} substring hits overruled")
    if ver:
        log("   these would have been counted as terms produced when they were "
            "words appearing in ordinary text:")
        for c in ver[:5]:
            log(f"      {c.get('n_ok_substring')} -> {c['n_ok']} terms")

# where the accepted answers came from, which says how much rests on the weak route
w = collections.Counter(c.get("where") for c in certs if c["passed"])
log(f"\nhow certified answers were found: {dict(w)}")
if w.get("in-order"):
    log("   in-order is the substring route, the one the scoring judge exists to")
    log("   check. Answers found only that way are the least reliable.")

# What the run cost. GPU seconds are wall clock with the model loaded.
#
# This is written BEFORE the push, and the order matters. Nothing is read back
# from the working directory: the next run seeds ctest_runs.csv from the
# DATASET, so a row appended after the push is not merely late, it is lost. The
# generator and the unquestionability notebook both do it in this order; this
# one did not, and its rows went missing.
if "REF" in globals() and hasattr(REF, "_loaded_at"):
    _GPU_SECONDS[0] = time.time() - REF._loaded_at
account(units=len(V8), note=f"metrics for {MACHINE_ID}")

# Persist the state, if this is Kaggle and credentials are set.
state_save(f"{BUILD}: metrics for {MACHINE_ID}")


In [ ]:
# ===========================================================================
# UNDO THE EXTRA HINT ROUND
# ===========================================================================
# Set HINTX_UNDO = True and run this cell to put the metrics back as they were.
# The pass writes `hintx` on the machine slot and `hint-x` programs carrying
# `cert_hintx`, and nothing else: K_cond, p_star, band, h and tau are never
# assigned, so Kt does not move whether the pass succeeded, failed or stopped
# half way. This removes the two things it does add.

HINTX_UNDO = False

if HINTX_UNDO:
    _n_rec = _n_prog = 0
    for _i, _v in (M.get("items") or {}).items():
        for _mid, _S in (_v or {}).items():
            if not isinstance(_S, dict): continue
            if _S.pop("hintx", None) is not None: _n_rec += 1
            _pg = _S.get("programs") or {}
            for _pid in [k for k, r in _pg.items() if r.get("tag") == "hint-x"]:
                _pg.pop(_pid); _n_prog += 1
    save()
    log(f"removed {_n_rec} hintx record(s) and {_n_prog} hint-x program(s)")
    log("nothing else was touched, so K_cond, band, h and tau are unchanged")
